# Spectral Decomposition Approach


Simplifed noteebook demonstrating night sky decomposition approach.

Sky spectrum decomposed into model of airglow sky lines and continuum components:

I. Continuum components include:

- Moon (high-resolution solar spectrum rebinned to LVM sampling and convolved to LVM Gaussian LSF and multiplied by B-spline multiplicative continuum). This mimicks Moon itself and Zodi components.

- Diffuse components (interpolated from low-resolution PALACE diffuse continuum components)

  - Hydroperoxyl ($HO_2$): This is the predominant continuum component in the near-infrared range, characterized by a prominent emission peak at 1.51 µm.

  - Iron Monoxide ($FeO$) and other molecules: This component dominates the visual wavelength range (roughly 500 to 720 nm) and includes the $FeO$ "orange arc" bands, with potential additional contributions from $NiO$ or $OFeOH$.

  - Unresolved Molecular Oxygen ($O_2$): Located in the ultraviolet (UVB) range, this component accounts for weak, unresolved bands from high-energy electronic states (specifically $c^1\Sigma^-_u$, $A'^3\Delta_u$, and $A^3\Sigma^+_u$).


II. Airglow sky components include:

  - Atomic Oxygen (O I) emission lines within the visual wavelength range.

  - Sodium (Na I): doublet at 5889.95 and 5895.92 Å, formed in the mesospheric Na layer at about 92 km by chemiluminescent reactions of meteoric sodium; typical D2 / D1 ≈ 1.7.
  
  - Potassium (K I): doublet at 7664.90 and 7698.96 Å, formed in the mesospheric K layer at about 89 km by chemistry similar to Na; typical D2 / D1 ≈ 1.67.
  
  - Nitrogen (N I): [ N I ] [NI] doublet at 5197.90 and 5200.26 Å, formed higher in the ionosphere at about 250 km via dissociative recombination; typical 5198 / 5200 ≈ 1.76.


  - Unresolved Molecular Oxygen ($O_2$): Located in the ultraviolet (UVB) range, this component accounts for weak, unresolved bands from high-energy electronic states (specifically $c^1\Sigma^-_u$, $A'^3\Delta_u$, and $A^3\Sigma^+_u$).

  - Hydroxyl (OH): This component accounts for the hydroxyl emission lines in the visual wavelength range (roughly 500 to 720 nm).

# Manual in-place experiments -- ND

## Imports and helpers

Load the core packages and define the helper functions used throughout the notebook.

In [8]:

import plotly.graph_objects as go
from astropy.io import fits
from astropy.table import Table
import numpy as np

FACTOR = 1e14
LSF_SIGMA = 0.5
T_O2 = 191.5 # in K

PALACE_DIR = ''
MEDIAN_STACK_DIR = ''

In [49]:
f = MEDIAN_STACK_DIR+'lvmsframe_median_stack_1.2.1_limit100.fits'
fits.info(f)

wave = fits.getdata(f, "WAVE").astype(np.float64)
flx_sky1 = fits.getdata(f, "FLUX_SKY_NEAR").astype(np.float64) * FACTOR
flx_sky2 = fits.getdata(f, "FLUX_SKY_FAR").astype(np.float64) * FACTOR
flx_sci = fits.getdata(f, "FLUX_SCI").astype(np.float64) * FACTOR
meta = Table(fits.getdata(f, "META"))
flx_ivar = 1.0 + np.zeros_like(flx_sci)  # fits.getdata(f, "FLUX_IVAR").astype(np.float64)
# flx_err = 1.0 / np.sqrt(flx_ivar)
# flx_sci = flx_flux + flx_sky
flx_sci.shape


def plot_fit_result(result, idx):
    bestfit = result.bestfit
    bestfit_lsf = result.bestfit_lsf
    comp_Moon = result.components["moon"]
    comp_DIFFUSE = result.components["diffuse"]

    resid = flx_sci[idx] - bestfit
    resid_lsf = flx_sci[idx] - bestfit_lsf
    resid_level = -3.0 * np.nanstd(resid)

    err_plot = 1.0 / np.sqrt(np.where(flx_ivar[idx] > 0, flx_ivar[idx], np.nan))

    fig = go.Figure()
    fig.add_trace(go.Scattergl(x=wave, y=flx_sci[idx], mode="lines", name="Observed",
                            line=dict(color="black", width=1)))
    fig.add_trace(go.Scattergl(x=wave, y=bestfit_lsf, mode="lines", name="Best-fit + LSF",
                            line=dict(color="darkorange", width=1.5)))
    fig.add_trace(go.Scattergl(x=wave, y=comp_Moon, mode="lines", name="Moon",
                            line=dict(color="#4e79a7", width=1, dash="dash")))

    knot_x = np.asarray(getattr(result, "moon_knots", []), dtype=float)
    knot_mask = np.isfinite(knot_x) & (knot_x >= wave[0]) & (knot_x <= wave[-1])
    knot_x = knot_x[knot_mask]
    if knot_x.size > 0:
        knot_y = np.interp(knot_x, wave, comp_Moon)
        fig.add_trace(go.Scattergl(
            x=knot_x,
            y=knot_y,
            mode="markers",
            name="Moon knots",
            marker=dict(symbol="x", size=8, color="#1f4e79", line=dict(width=1)),
        ))

    boosted_x = np.asarray(getattr(result, "moon_boosted_pixels", []), dtype=float)
    boosted_mask = np.isfinite(boosted_x) & (boosted_x >= wave[0]) & (boosted_x <= wave[-1])
    boosted_x = boosted_x[boosted_mask]
    if boosted_x.size > 0:
        boosted_y = np.interp(boosted_x, wave, comp_Moon)
        fig.add_trace(go.Scattergl(
            x=boosted_x,
            y=boosted_y,
            mode="markers",
            name="Moon boosted pixels",
            marker=dict(symbol="circle-open", size=5, color="#e15759", line=dict(width=1)),
        ))

    fig.add_trace(go.Scattergl(x=wave, y=comp_DIFFUSE, mode="lines", name="Diffuse",
                            line=dict(color="#59a14f", width=1, dash="dash")))
    fig.add_trace(go.Scattergl(x=wave, y=resid_level + resid_lsf, mode="lines", name="Residual + LSF",
                            line=dict(color="seagreen", width=1)))
    fig.add_trace(go.Scattergl(x=wave, y=resid_level + err_plot, mode="lines", name="+1sigma",
                            line=dict(color="gray", width=1, dash="dash")))
    fig.add_trace(go.Scattergl(x=wave, y=resid_level - err_plot, mode="lines", name="-1sigma",
                            line=dict(color="gray", width=1, dash="dash")))

    fig.update_layout(
        title={
            "text": (
                f"idx={idx} | T_O2={result.t_o2:.1f}±{result.t_o2_err:.1f} K<br>"
                f"{result.fit_summary}"
            ),
            "font": {"size": 14},
        },
        xaxis_title="λ (Å)",
        yaxis_title="Flux",
        template="plotly_white",
        height=520,
    )
    fig.show()



Filename: lvmsframe_median_stack_1.2.1_limit100.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU      12   ()      
  1  WAVE          1 ImageHDU         9   (12401,)   float32   
  2  FLUX_SCI      1 ImageHDU        10   (12401, 100)   float32   
  3  FLUX_SKY_NEAR    1 ImageHDU        10   (12401, 100)   float32   
  4  FLUX_SKY_FAR    1 ImageHDU        10   (12401, 100)   float32   
  5  FLUX_SCI_NOSKY    1 ImageHDU        10   (12401, 100)   float32   
  6  LSF_SCI       1 ImageHDU        10   (12401, 100)   float32   
  7  LSF_SKY_NEAR    1 ImageHDU        10   (12401, 100)   float32   
  8  LSF_SKY_FAR    1 ImageHDU        10   (12401, 100)   float32   
  9  META          1 BinTableHDU    107   100R x 48C   [512A, K, K, K, K, 32A, 32A, D, D, D, D, D, D, 8A, 8A, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, K, K, K, K, K, K, K, 16A, 512A]   


In [58]:
import importlib
import numpy as np
import sky_decomp.fit as fitmod

importlib.reload(fitmod)
SkyDecomp = fitmod.SkyDecomp

decomposer = SkyDecomp(
    wave,
    lsf_sigma=LSF_SIGMA,
    moon_smooth_lambda=0.1,
    moon_interline_boost=10000.0,
    moon_interline_red_min=6000.0,
    moon_interline_exclusion_a=2.5,
    moon_interline_line_flux_threshold=0.01,
 )

idx = 20
result = decomposer.fit(
    flx_sci[idx],
    flx_ivar[idx],
    verbose=True,
    n_lsf_refits=3,
 )
plot_fit_result(result, idx)


O2
  T         171.50 +/- 5.95 K
  chi2_red  0.1816
  dt        0.212 s

decomp
  init      2.069

iterations
  [1] LSF     B=0.4468   R=1.277   Z=13.97   dt=0.002 s
      decomp  chi2_red=2.458   qp=0.180 s
  [2] LSF     B=0.07016   R=0.4547   Z=4.199   dt=0.002 s
      decomp  chi2_red=1.375   qp=0.182 s
  [3] LSF     B=0.04261   R=0.3295   Z=3.007   dt=0.002 s
      decomp  chi2_red=1.236   qp=0.177 s

final
  decomp    1.236
  refits    3
  total_dt  1.768 s
  peak_mem  252.09 MB


In [59]:
# Multi-index stability check (no moon-refit iteration sweep)
import importlib
import numpy as np
import pandas as pd
import sky_decomp.fit as fitmod

importlib.reload(fitmod)
SkyDecomp = fitmod.SkyDecomp

idx_values = list(range(10, 31))
red_mask = wave >= 6000.0

rows = []
for idx_sweep in idx_values:
    decomp_try = SkyDecomp(
        wave,
        lsf_sigma=LSF_SIGMA,
        moon_smooth_lambda=0.1,
        moon_interline_boost=10000.0,
        moon_interline_red_min=6000.0,
        moon_interline_exclusion_a=2.5,
        moon_interline_line_flux_threshold=0.01,
    )
    res_try = decomp_try.fit(
        flx_sci[idx_sweep],
        flx_ivar[idx_sweep],
        verbose=False,
        n_lsf_refits=3,
    )

    resid_lsf = flx_sci[idx_sweep] - res_try.bestfit_lsf
    red_rms = float(np.sqrt(np.nanmean(resid_lsf[red_mask] ** 2)))

    boosted = np.asarray(getattr(res_try, "moon_boosted_pixels", []), dtype=float)
    if boosted.size > 0:
        pix_idx = np.searchsorted(wave, boosted)
        pix_idx = np.clip(pix_idx, 0, wave.size - 1)
        boosted_rms = float(np.sqrt(np.nanmean(resid_lsf[pix_idx] ** 2)))
        n_boosted = int(boosted.size)
    else:
        boosted_rms = np.nan
        n_boosted = 0

    rows.append({
        "idx": int(idx_sweep),
        "chi2_red": float(res_try.reduced_chi2),
        "r2": float(res_try.r2),
        "red_rms": red_rms,
        "boosted_rms": boosted_rms,
        "n_boosted": n_boosted,
        "fit_elapsed_sec": float(res_try.fit_elapsed_sec),
    })

stability_df = pd.DataFrame(rows).sort_values("idx").reset_index(drop=True)

print("Per-index metrics with single moon+diffuse refit:")
display(stability_df)

summary = {
    "n_idx": int(stability_df.shape[0]),
    "chi2_median": float(stability_df["chi2_red"].median()),
    "chi2_p90": float(np.nanpercentile(stability_df["chi2_red"], 90)),
    "red_rms_median": float(stability_df["red_rms"].median()),
    "red_rms_p90": float(np.nanpercentile(stability_df["red_rms"], 90)),
    "runtime_median_sec": float(stability_df["fit_elapsed_sec"].median()),
}
print("\nSummary:")
print(summary)

single_refit_stability = stability_df
single_refit_summary = summary

Per-index metrics with single moon+diffuse refit:


,idx,chi2_red,r2,red_rms,boosted_rms,n_boosted,fit_elapsed_sec
0,10,1.820387e+00,8.868585e-01,1.657974e+00,9.410504e-01,2280,1.753745
1,11,1.704302e+00,8.884458e-01,1.627210e+00,9.353737e-01,2280,1.718276
2,12,1.978346e+28,-1.842275e-01,1.698450e+14,1.706197e+14,7419,1.596709
3,13,3.928413e+42,-2.051435e+14,9.857989e+19,9.883147e+19,7419,1.578936
4,14,2.583307e+28,-6.324454e-02,1.944378e+14,1.953490e+14,7419,1.827727
5,15,4.073791e+32,-1.801265e+04,2.528294e+16,2.513145e+16,7419,1.541114
6,16,1.393565e+00,8.896774e-01,1.462685e+00,3.338314e-01,2280,1.911179
7,17,1.513277e+00,8.896738e-01,1.526561e+00,3.300369e-01,2280,10.644258
8,18,1.643399e+00,8.887750e-01,1.590130e+00,3.455223e-01,2280,5.737745
9,19,1.177248e+00,8.994429e-01,1.354792e+00,1.360186e-01,2280,5.918337



Summary:
{'n_idx': 21, 'chi2_median': 1.5132771470020447, 'chi2_p90': 2.5833073619411894e+28, 'red_rms_median': 1.5265609039177377, 'red_rms_p90': 194437830871498.75, 'runtime_median_sec': 1.7821613749983953}


In [57]:
# Inspect outlier indices from the multi-index sweep
import numpy as np

q_bad = np.nanpercentile(sweep_results_detail["chi2_red"], 95)
bad = sweep_results_detail[sweep_results_detail["chi2_red"] >= q_bad].copy()
bad = bad.sort_values(["idx", "chi2_red"]).reset_index(drop=True)

print(f"95th percentile chi2_red threshold: {q_bad:.6g}")
print("Rows above threshold:")
display(bad[["idx", "refit_iters", "min_rel_gain", "chi2_red", "red_rms", "fit_elapsed_sec"]])

print("\nPer-index median chi2_red (across settings):")
per_idx = (
    sweep_results_detail.groupby("idx", as_index=False)["chi2_red"]
    .median()
    .sort_values("chi2_red", ascending=False)
    .reset_index(drop=True)
)
display(per_idx.head(10))

95th percentile chi2_red threshold: 2.00282e+28
Rows above threshold:


,idx,refit_iters,min_rel_gain,chi2_red,red_rms,fit_elapsed_sec
0,12,1,0.010,1.132142e+30,1.332906e+15,1.500938
1,12,1,0.005,1.132142e+30,1.332906e+15,1.510625
2,12,1,0.001,1.132142e+30,1.332906e+15,1.512546
3,12,2,0.010,1.132142e+30,1.332906e+15,1.524072
4,12,2,0.005,1.132142e+30,1.332906e+15,1.517659
5,12,2,0.001,1.132142e+30,1.332906e+15,1.563155
6,12,3,0.010,1.132142e+30,1.332906e+15,1.523224
7,12,3,0.005,1.132142e+30,1.332906e+15,1.520817
8,12,3,0.001,1.132142e+30,1.332906e+15,1.520392
9,15,1,0.010,2.002816e+28,1.757931e+14,1.689409



Per-index median chi2_red (across settings):


,idx,chi2_red
0,12,1.132142e+30
1,15,2.002816e+28
2,14,2.921784e+27
3,13,2.299934e+27
4,28,1.774627e+04
5,29,2.391548e+00
6,30,2.026432e+00
7,10,1.783079e+00
8,11,1.672950e+00
9,18,1.603562e+00


In [22]:
def thin_fits_every_n(input_path, output_path, n, row_hdu_name="META"):
    """Write a new FITS with every n-th row-like element kept.

    The function preserves HDU structure and headers. It identifies the row
    count from `row_hdu_name` (default: META), then slices any table HDU with
    that row count and any image HDU whose first axis matches that row count.
    """
    if n < 1:
        raise ValueError("n must be >= 1")

    with fits.open(input_path) as hdul:
        if row_hdu_name not in hdul:
            raise KeyError(f"HDU '{row_hdu_name}' not found in {input_path}")

        n_rows = len(hdul[row_hdu_name].data)
        keep = slice(None, None, n)

        out_hdus = []
        for hdu in hdul:
            header = hdu.header.copy()

            if isinstance(hdu, fits.PrimaryHDU):
                data = hdu.data
                if data is not None and getattr(data, "ndim", 0) >= 1 and data.shape[0] == n_rows:
                    data = data[keep, ...]
                out_hdus.append(fits.PrimaryHDU(data=data, header=header))

            elif isinstance(hdu, (fits.BinTableHDU, fits.TableHDU)):
                data = hdu.data
                if data is not None and len(data) == n_rows:
                    data = data[keep]
                out_hdus.append(type(hdu)(data=data, header=header, name=hdu.name))

            elif isinstance(hdu, (fits.ImageHDU, fits.CompImageHDU)):
                data = hdu.data
                if data is not None and getattr(data, "ndim", 0) >= 1 and data.shape[0] == n_rows:
                    data = data[keep, ...]
                out_hdus.append(type(hdu)(data=data, header=header, name=hdu.name))

            else:
                out_hdus.append(hdu.copy())

        fits.HDUList(out_hdus).writeto(output_path, overwrite=True)


# Example:
# thin_fits_every_n(
#     "../lvmsframe_median_stack_1.2.1_limit100.fits",
#     "../lvmsframe_median_stack_1.2.1_limit100_every5.fits",
#     n=5)

def results_to_fits(results, filename):
    """Write a list of SkyDecompResult objects to a FITS file.
    
    Scalar quantities go into a binary table (extension META).
    Spectral/coefficient arrays go into separate image extensions.
    
    Extensions:
        META         - BinTable with scalar fields per result
        COEF         - (n_results, n_coef) fit coefficients
        BESTFIT      - (n_results, n_wave) initial best-fit spectra
        BESTFIT_LSF  - (n_results, n_wave) LSF-refined best-fit spectra
        RESID        - (n_results, n_wave) residuals
        COMP_<KEY>   - (n_results, n_wave) per component spectra
    """
    rows = {
        "t_o2":               [r.t_o2 for r in results],
        "t_o2_err":           [r.t_o2_err for r in results],
        "reduced_chi2":       [r.reduced_chi2 for r in results],
        "r2":                 [r.r2 for r in results],
        "rms_resid":          [r.rms_resid for r in results],
        "resid_level":        [r.resid_level for r in results],
        "fit_status":         [r.fit_status for r in results],
        "fit_summary":        [r.fit_summary for r in results],
        "fit_elapsed_sec":    [r.fit_elapsed_sec for r in results],
        "peak_memory_mb":     [r.peak_memory_mb for r in results],
        "o2_fit_status":      [r.o2_fit_status for r in results],
        "o2_fit_summary":     [r.o2_fit_summary for r in results],
        "o2_fit_elapsed_sec": [r.o2_fit_elapsed_sec for r in results],
        "o2_valid_frac":      [r.o2_valid_frac for r in results],
    }
    t = Table(rows)

    def stack(attr):
        return np.vstack([getattr(r, attr) for r in results])

    # COEF header: store design_names as FITS keywords for reference
    coef_arr = stack("coef")
    coef_hdu = fits.ImageHDU(coef_arr, name="COEF")
    design_names = results[0].design_names
    for i, name in enumerate(design_names):
        coef_hdu.header[f"COEF{i:04d}"] = name

    hdul = fits.HDUList([
        fits.PrimaryHDU(),
        fits.BinTableHDU(t, name="META"),
        coef_hdu,
        fits.ImageHDU(stack("bestfit"),     name="BESTFIT"),
        fits.ImageHDU(stack("bestfit_lsf"), name="BESTFIT_LSF"),
        fits.ImageHDU(stack("resid"),       name="RESID"),
    ])

    comp_keys = list(results[0].components.keys())
    for key in comp_keys:
        arr = np.vstack([r.components[key] for r in results])
        hdul.append(fits.ImageHDU(arr, name=f"COMP_{key.upper()}"))

    hdul.writeto(filename, overwrite=True)
    print(f"Wrote {len(results)} results, {coef_arr.shape[1]} coefs, {len(comp_keys)} components -> {filename}")


def _copy_hdu_with_name(hdu, extname):
    """Return a copy of an HDU with a new extension name."""
    header = hdu.header.copy()
    header["EXTNAME"] = extname
    return type(hdu)(data=hdu.data, header=header, name=extname)


def _infer_decomp_label(path, index):
    """Infer a stable label for a decomposition file from its filename."""
    from pathlib import Path

    name = Path(path).name.lower()
    for label in ("sky1", "sky2", "sci"):
        if label in name:
            return label.upper()
    return f"DEC{index}"


def extract_meta_and_coef_products(
    input_fits_path,
    decomp_fits_path_1,
    decomp_fits_path_2,
    decomp_fits_path_3,
    meta_output_path=None,
    sky1_output_path=None,
    sky2_output_path=None,
    sci_output_path=None,
):
    """Write compact FITS products containing only selected extensions.

    The first output contains only the META extension from `input_fits_path`.
    Each decomposition input gets its own output FITS containing just META and
    COEF. Default output paths are written in the current working directory.
    """
    from pathlib import Path

    input_path = Path(input_fits_path)
    cwd = Path.cwd()
    if meta_output_path is None:
        meta_output_path = str(cwd / f"{input_path.stem}_meta_only{input_path.suffix}")

    decomp_files = [decomp_fits_path_1, decomp_fits_path_2, decomp_fits_path_3]
    decomp_outputs = [sky1_output_path, sky2_output_path, sci_output_path]

    with fits.open(input_fits_path) as hdul_in:
        if "META" not in hdul_in:
            raise KeyError(f"Missing META extension in {input_fits_path}")
        fits.HDUList([
            fits.PrimaryHDU(),
            _copy_hdu_with_name(hdul_in["META"], "META"),
        ]).writeto(meta_output_path, overwrite=True)

    resolved_outputs = []
    for index, decomp_path in enumerate(decomp_files, start=1):
        label = _infer_decomp_label(decomp_path, index)
        out_path = decomp_outputs[index - 1]
        if out_path is None:
            out_path = str(cwd / f"{input_path.stem}_{label.lower()}_meta_coef{input_path.suffix}")
        with fits.open(decomp_path) as hdul_dec:
            for extname in ("META", "COEF"):
                if extname not in hdul_dec:
                    raise KeyError(f"Missing {extname} extension in {decomp_path}")
            fits.HDUList([
                fits.PrimaryHDU(),
                _copy_hdu_with_name(hdul_dec["META"], "META"),
                _copy_hdu_with_name(hdul_dec["COEF"], "COEF"),
            ]).writeto(out_path, overwrite=True)
        print(f"Wrote {label} META/COEF file -> {out_path}")
        resolved_outputs.append(out_path)

    print(f"Wrote META-only file -> {meta_output_path}")
    return (meta_output_path, *resolved_outputs)

In [23]:
#hdul = fits.open("lvmsframe_median_stack_1.2.1_every10_decomp_sci.fits")
hdul = fits.open("lvmsframe_median_stack_1.2.1_limit100.fits")
hdul.info()
#t = Table(hdul["COEF"].data)
#t.colnames[-20:]
t = Table(hdul["META"].data)
#t.colnames
t

Filename: lvmsframe_median_stack_1.2.1_limit100.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU      12   ()      
  1  WAVE          1 ImageHDU         9   (12401,)   float32   
  2  FLUX_SCI      1 ImageHDU        10   (12401, 100)   float32   
  3  FLUX_SKY_NEAR    1 ImageHDU        10   (12401, 100)   float32   
  4  FLUX_SKY_FAR    1 ImageHDU        10   (12401, 100)   float32   
  5  FLUX_SCI_NOSKY    1 ImageHDU        10   (12401, 100)   float32   
  6  LSF_SCI       1 ImageHDU        10   (12401, 100)   float32   
  7  LSF_SKY_NEAR    1 ImageHDU        10   (12401, 100)   float32   
  8  LSF_SKY_FAR    1 ImageHDU        10   (12401, 100)   float32   
  9  META          1 BinTableHDU    107   100R x 48C   [512A, K, K, K, K, 32A, 32A, D, D, D, D, D, D, 8A, 8A, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, K, K, K, K, K, K, K, 16A, 512A]   


path,exposure,expnum,mjd,tile_id,obstime,date_obs,sci_ra,sci_dec,skye_ra,skye_dec,skyw_ra,skyw_dec,sky_near_label,sky_far_label,sky_near_ra,sky_near_dec,sky_far_ra,sky_far_dec,skye_sep_deg,skyw_sep_deg,sky_near_sep_deg,sky_far_sep_deg,sci_alt,skye_alt,skyw_alt,sci_airmass,skye_airmass,skyw_airmass,sci_moon_sep,skye_moon_sep,skyw_moon_sep,moon_alt,sun_alt,moon_ra,moon_dec,moon_phase,moon_fli,moon_illum,n_fib_sci_good,n_fib_skye_good,n_fib_skyw_good,n_fib_sci_used,n_fib_skye_used,n_fib_skyw_used,worker_pid,status,error
str512,int64,int64,int64,int64,str32,str32,float64,float64,float64,float64,float64,float64,str8,str8,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64,int64,int64,int64,int64,int64,int64,str16,str512
/data/sas/sdsswork/lvm/spectro/redux/1.2.1/0011XX/11111/60177/lvmSFrame-00003469.fits,3469,-1,60177,11111,2023-08-21T03:03:19.735,2023-08-21T03:03:19.735,269.8838,-23.3863,269.631147,-26.136676,264.909766,-19.952604,SkyE,SkyW,269.631147,-26.136676,264.909766,-19.952604,2.7599,5.7573,2.7599,5.7573,59.0184,59.5447,53.4951,1.1664084030530644,1.1600594825071733,1.2440820347700314,71.4703,71.3048,66.7809,-11.1825,-60.7981,197.28013,-6.5329,50.92,0.1856,0.1856,1754,57,60,1228,40,42,180127,OK,
/data/sas/sdsswork/lvm/spectro/redux/1.2.1/0011XX/11111/60177/lvmSFrame-00003470.fits,3470,-1,60177,11111,2023-08-21T03:05:20.788,2023-08-21T03:05:20.788,269.88380853902834,-23.386325253537432,269.6311308897608,-26.136693543338072,264.9097495419595,-19.95263414746738,SkyE,SkyW,269.6311308897608,-26.136693543338072,264.9097495419595,-19.95263414746738,2.7599,5.7573,2.7599,5.7573,58.5765,59.103,53.0546,1.171870342032079,1.1653765532379894,1.2512381850967453,71.4517,71.2862,66.7623,-11.5927,-61.1796,197.29743,-6.53995,50.94,0.1857,0.1857,1754,57,60,1228,40,42,180128,OK,
/data/sas/sdsswork/lvm/spectro/redux/1.2.1/0011XX/11111/60177/lvmSFrame-00003471.fits,3471,-1,60177,11111,2023-08-21T03:23:19.400,2023-08-21T03:23:19.400,269.8838,-23.3863,269.631147,-26.136676,264.909766,-19.952604,SkyE,SkyW,269.631147,-26.136676,264.909766,-19.952604,2.7599,5.7573,2.7599,5.7573,54.6363,55.1718,49.1224,1.2262492655097017,1.2182220567943667,1.3225590652160537,71.2839,71.1187,66.5948,-15.2224,-64.4536,197.45406,-6.60277,51.1,0.1868,0.1868,1754,57,60,1228,40,42,180129,OK,
/data/sas/sdsswork/lvm/spectro/redux/1.2.1/0011XX/11111/60177/lvmSFrame-00003472.fits,3472,-1,60177,11111,2023-08-21T03:25:20.245,2023-08-21T03:25:20.245,269.8838005747448,-23.38626647284794,269.63108626978044,-26.136695220783917,264.9097219509721,-19.952630251193426,SkyE,SkyW,269.63108626978044,-26.136695220783917,264.9097219509721,-19.952630251193426,2.76,5.7573,2.76,5.7573,54.1948,54.7319,48.6813,1.2330295249387466,1.224801020311552,1.3314722303616273,71.2649,71.0996,66.5757,-15.626,-64.804,197.47188,-6.60981,51.11,0.1869,0.1869,1754,57,60,1228,40,42,180130,OK,
/data/sas/sdsswork/lvm/spectro/redux/1.2.1/0011XX/11111/60177/lvmSFrame-00003473.fits,3473,-1,60177,11111,2023-08-21T03:43:38.458,2023-08-21T03:43:38.458,269.8838,-23.3863,269.631147,-26.136676,264.909766,-19.952604,SkyE,SkyW,269.631147,-26.136676,264.909766,-19.952604,2.7599,5.7573,2.7599,5.7573,50.1838,50.7409,44.6697,1.301910140629899,1.2915030110027814,1.4224362217804565,71.0894,70.9247,66.4004,-19.2614,-67.7907,197.63635,-6.67384,51.28,0.188,0.188,1754,57,60,1228,40,42,180130,OK,
/data/sas/sdsswork/lvm/spectro/redux/1.2.1/0011XX/11111/60177/lvmSFrame-00003474.fits,3474,-1,60177,11111,2023-08-21T03:45:40.135,2023-08-21T03:45:40.135,269.8838278853023,-23.386306835086867,269.63081411738676,-26.13641953092062,264.90976524130747,-19.952620530933718,SkyE,SkyW,269.63081411738676,-26.13641953092062,264.90976524130747,-19.952620530933718,2.7597,5.7573,2.7597,5.7573,49.7397,50.2991,44.2252,1.3104165527918479,1.2997319775091907,1.4337327906458919,71.0698,70.9047,66.3808,-19.6603,-68.096,197.

## VAE Modeling Experiments

## Current Model Design

The main VAE cell below is no longer the original baseline only. It now contains the active model used for the current experiments.

Key changes relative to the original notebook version:

1. Latent size is reduced to `z_dim = 8`. The earlier latent-capacity check showed that the effective latent dimensionality was well below 16, so the model now uses a smaller latent state to reduce wasted capacity and discourage overly flexible latent memorization.

2. Regularization is modestly stronger than the original low-beta run. The current training setup uses a larger KL weight ceiling and slightly stronger latent-mean regularization so that the decoder must rely more on context and less on unconstrained latent variation.

3. Correlated low-rank heads are used for structured coefficient families.
- A spline head models the strongly correlated `Moon_bs*` block.
- An OH head models the strongly correlated `OH_*` block.

4. The decoder now has a deterministic context-conditioned continuum path. This is important because notebook inference uses a deterministic decode path (`z = 0` for row-wise prediction from metadata). The additional context-only continuum head makes continuum coefficients depend directly on metadata instead of requiring latent information that is unavailable at deterministic inference time.

5. The old post-hoc OH correction stage is no longer part of the model logic. A compatibility no-op remains so downstream notebook cells continue to run without having to special-case older helper calls.

6. Device selection is conservative on Apple Silicon. The notebook now prefers CPU by default and only uses MPS when `batch_size > 256`, because a representative benchmark on this machine showed that MPS is slower than CPU for the current small-MLP / small-batch workload.

In [24]:
# Code map for the main model cell below:

# 1) Data assembly and filtering
#    - read_decomp_dataset(...) loads coefficient matrices and aligned metadata context.
#    - chi2 / hard-clip / kappa-sigma filters reduce pathological rows before training.

# 2) Base helpers
#    - RobustScaler keeps coefficient and context normalization robust to heavy tails.
#    - _coef_to_model_space / _coef_from_model_space apply the sqrt / square transform used for non-negative coefficients.

# 3) Active model architecture
#    - The encoder maps (coef, context) -> (mu, logvar).
#    - The main decoder path uses (z, context) to predict all coefficients.
#    - A low-rank spline head adds structured corrections to the Moon_bs block.
#    - A low-rank OH head adds structured corrections to the OH block.
#    - A context-only continuum head adds deterministic corrections to continuum-like coefficients so inference from metadata does not rely entirely on latent variables.

# 4) Loss terms
#    - reconstruction term: smooth-L1 on normalized/model-space coefficients
#    - KL term: weighted by beta with warmup
#    - latent L2 term: weak penalty on latent mean magnitude
#    - spline smoothness term: second-difference penalty on the spline block
#    - OH smoothness term is currently disabled (weight = 0.0)

# 5) Inference path used elsewhere in the notebook
#    - Row-wise prediction cells decode with z = 0 and context only.
#    - That is why the deterministic continuum context head matters: it pushes continuum structure into the metadata-conditioned path that is actually used at prediction time.

In [2]:
# Missing utility helpers required by the main ND training cell
import random
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset


def _set_reproducibility(seed=42, deterministic=False):
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def _finite_report(name, x):
    arr = np.asarray(x)
    frac = float(np.isfinite(arr).mean()) if arr.size > 0 else 1.0
    print(f"{name}: shape={arr.shape}, finite={100.0 * frac:.2f}%")


def split_indices(n, train_frac=0.8, val_frac=0.1, seed=42):
    rng = np.random.default_rng(int(seed))
    idx = np.arange(int(n))
    rng.shuffle(idx)
    n_train = int(train_frac * n)
    n_val = int(val_frac * n)
    train_idx = idx[:n_train]
    val_idx = idx[n_train:n_train + n_val]
    test_idx = idx[n_train + n_val:]
    return train_idx, val_idx, test_idx


def make_loader(coef_n, ctx_n, idx, batch_size=256, shuffle=False, generator=None):
    x_coef = torch.from_numpy(np.asarray(coef_n[idx], dtype=np.float32))
    x_ctx = torch.from_numpy(np.asarray(ctx_n[idx], dtype=np.float32))
    ds = TensorDataset(x_coef, x_ctx)
    return DataLoader(ds, batch_size=int(batch_size), shuffle=bool(shuffle), drop_last=False, generator=generator)


def _per_dim_quantile_bounds(x, q_low=0.1, q_high=99.9):
    arr = np.asarray(x, dtype=np.float64)
    lo = np.nanpercentile(arr, float(q_low), axis=0).astype(np.float32)
    hi = np.nanpercentile(arr, float(q_high), axis=0).astype(np.float32)
    bad = ~(np.isfinite(lo) & np.isfinite(hi) & (hi > lo))
    if np.any(bad):
        lo2 = np.nanmin(arr, axis=0).astype(np.float32)
        hi2 = np.nanmax(arr, axis=0).astype(np.float32)
        lo[bad] = lo2[bad]
        hi[bad] = hi2[bad]
    span = np.maximum(hi - lo, 1e-6)
    hi = lo + span
    return lo, hi

In [3]:
# V1 baseline (stable): conditional VAE for decomposition coefficients
# This version models p(coef | context) with guarded training to avoid NaNs.
from astropy.io import fits
from astropy.table import Table
import numpy as np
import pandas
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import plotly.express as px
from torch.utils.data import TensorDataset, DataLoader
from sklearn.linear_model import Ridge


def _as_array(x):
    """Return numeric input as float32 array; skip non-numeric columns.

    This helper is used when reading FITS tables that can contain string/object
    metadata columns mixed with numeric fields.
    """
    arr = np.asarray(x)
    if arr.dtype.kind in ("U", "S", "O"):
        # Non-numeric fields are intentionally ignored in this baseline.
        return None
    return arr.astype(np.float32)


def _coerce_coef_hdu_to_table(coef_hdu):
    """Normalize COEF extension into an Astropy table.

    Supports both historical layouts:
    1) `BinTableHDU`/`TableHDU` with named columns.
    2) `ImageHDU` (2D array) where names may be stored in `COEFxxxx` header keys.
    """
    data = coef_hdu.data
    if isinstance(coef_hdu, (fits.BinTableHDU, fits.TableHDU)):
        return Table(data)
    # ImageHDU fallback: infer names from COEFxxxx header keywords if available.
    arr = np.asarray(data, dtype=np.float32)
    if arr.ndim != 2:
        raise ValueError(f"Expected 2D COEF image, got shape={arr.shape}")
    n_coef = arr.shape[1]
    names = []
    for i in range(n_coef):
        key = f"COEF{i:04d}"
        names.append(str(coef_hdu.header.get(key, f"coef_{i:04d}")))
    return Table({name: arr[:, i] for i, name in enumerate(names)})


def _select_context_from_labels(meta, meta_upper, labels, base_name):
    """Build a single context vector using SKYE_/SKYW_ columns and row labels.

    For a requested base name (e.g., `moon_sep`), this function expects
    `SKYE_MOON_SEP` and `SKYW_MOON_SEP` columns. It then selects the right value
    per row according to label values (`SKYE` or `SKYW`).
    """
    e_key = f"SKYE_{base_name.upper()}"
    w_key = f"SKYW_{base_name.upper()}"
    if e_key not in meta_upper or w_key not in meta_upper:
        return None
    arr_e = _as_array(meta[meta_upper[e_key]])
    arr_w = _as_array(meta[meta_upper[w_key]])
    if arr_e is None or arr_w is None:
        raise ValueError(f"Labeled context columns for '{base_name}' are non-numeric.")
    is_e = labels == "SKYE"
    is_w = labels == "SKYW"
    if not np.all(is_e | is_w):
        bad = np.unique(labels[~(is_e | is_w)])
        raise ValueError(f"Unexpected label values: {bad}")
    out = np.where(is_e, arr_e, arr_w).astype(np.float32)
    return out


def _table_to_float32_matrix(tbl, value_name):
    """Convert numeric columns from an Astropy table into a float32 matrix."""
    names = list(tbl.colnames)
    cols = []
    for name in names:
        arr = _as_array(tbl[name])
        if arr is not None:
            cols.append(arr)
    if len(cols) == 0:
        raise ValueError(f"No numeric {value_name} columns found.")
    return np.column_stack(cols).astype(np.float32), names


def _build_context_matrix(meta, context_columns, kind):
    """Build context matrix from META columns and label-selected SKYE_/SKYW_ values."""
    meta_upper = {c.upper(): c for c in meta.colnames}
    labels = None
    if kind in ("sky1", "sky2"):
        label_col = "SKY_NEAR_LABEL" if kind == "sky1" else "SKY_FAR_LABEL"
        if label_col not in meta_upper:
            raise KeyError(f"Missing required META label column: {label_col}")
        labels = np.char.upper(np.char.strip(np.asarray(meta[meta_upper[label_col]]).astype(str)))

    ctx_names = []
    ctx_cols = []
    missing_cols = []
    for cname in context_columns:
        key = cname.upper()
        if key in meta_upper:
            arr = _as_array(meta[meta_upper[key]])
            if arr is None:
                raise ValueError(f"Context column '{cname}' is non-numeric.")
            ctx_names.append(cname)
            ctx_cols.append(arr)
            continue
        if labels is not None:
            arr = _select_context_from_labels(meta, meta_upper, labels, cname)
            if arr is not None:
                ctx_names.append(cname)
                ctx_cols.append(arr)
                continue
        missing_cols.append(cname)

    if missing_cols:
        raise KeyError(f"Missing requested context columns: {missing_cols}")
    if len(ctx_cols) == 0:
        raise ValueError("No usable context columns were assembled.")
    return np.column_stack(ctx_cols).astype(np.float32), ctx_names


def read_decomp_dataset(
    decomp_fits_path,
    input_fits_path,
    context_columns,
    decomp_kind="sky1",
    return_chi2=False,
):
    """Read decomposition coefficients and aligned context from FITS inputs.

    Parameters
    ----------
    decomp_fits_path : str
        FITS file produced by decomposition (must contain COEF extension).
    input_fits_path : str
        Original input FITS file (must contain META extension).
    context_columns : list[str]
        Context feature names. If a name is not found directly in META, the
        function tries `skye_<name>`/`skyw_<name>` and chooses per-row values
        based on near/far sky labels.
    decomp_kind : str
        One of {"sky1", "sky2", "sci"}. Controls which label column is used.
    return_chi2 : bool
        When True, also return the aligned reduced-chi^2 vector.

    Returns
    -------
    tuple
        `(coef_mat, ctx_mat, coef_names, ctx_names)` by default, or
        `(coef_mat, ctx_mat, coef_names, ctx_names, chi2_used)` when
        `return_chi2=True`.
    """
    if context_columns is None or len(context_columns) == 0:
        raise ValueError("context_columns must be a non-empty list of META column names.")
    kind = decomp_kind.lower()
    if kind not in ("sky1", "sky2", "sci"):
        raise ValueError("decomp_kind must be one of: 'sky1', 'sky2', 'sci'")

    with fits.open(decomp_fits_path) as hdul_dec, fits.open(input_fits_path) as hdul_in:
        coef_tbl = _coerce_coef_hdu_to_table(hdul_dec["COEF"])
        coef_mat, coef_names = _table_to_float32_matrix(coef_tbl, "coefficient")
        meta = Table(hdul_in["META"].data)
        ctx_mat, ctx_names = _build_context_matrix(meta, context_columns, kind)

        if coef_mat.shape[0] != ctx_mat.shape[0]:
            raise ValueError(
                f"Row count mismatch: COEF has {coef_mat.shape[0]} rows, META has {ctx_mat.shape[0]} rows"
            )

        good = np.isfinite(coef_mat).all(axis=1) & np.isfinite(ctx_mat).all(axis=1)
        coef_mat = coef_mat[good]
        ctx_mat = ctx_mat[good]

        if not return_chi2:
            return coef_mat, ctx_mat, coef_names, ctx_names

        dec_meta = Table(hdul_dec["META"].data)
        chi2_col = _find_chi2_column(dec_meta)
        chi2_full = np.asarray(dec_meta[chi2_col], dtype=np.float64)
        if chi2_full.shape[0] != good.shape[0]:
            raise ValueError(
                f"chi2 rows ({chi2_full.shape[0]}) do not match decomposition rows ({good.shape[0]})"
            )
        chi2_used = chi2_full[good]
        if chi2_used.shape[0] != coef_mat.shape[0]:
            raise ValueError(
                f"Aligned chi2 rows ({chi2_used.shape[0]}) do not match coef rows ({coef_mat.shape[0]})"
            )
        return coef_mat, ctx_mat, coef_names, ctx_names, chi2_used


def _find_chi2_column(meta_tbl):
    """Find a reduced-chi^2-like column in decomposition META table.

    The decomposition outputs can vary by naming convention across runs; this
    helper probes common aliases in priority order.
    """
    names = {c.upper(): c for c in meta_tbl.colnames}
    for cand in ["REDUCED_CHI2", "CHI2_REDUCED", "CHI2", "RCHI2"]:
        if cand in names:
            return names[cand]
    raise KeyError("No chi2-like column found in decomposition META table")


def _build_physics_group_ids(coef_names):
    """Map coefficient names to physics groups for learned embeddings."""
    group_names = ["moon", "oh", "diffuse", "atomic"]
    group_to_id = {name: i for i, name in enumerate(group_names)}
    group_ids = np.zeros(len(coef_names), dtype=np.int64)
    counts = {name: 0 for name in group_names}

    for i, name in enumerate(coef_names):
        lname = str(name).lower()
        if lname.startswith("moon_bs"):
            gname = "moon"
        elif lname.startswith("oh"):
            gname = "oh"
        elif lname.startswith("ho2") or lname.startswith("feo") or lname.startswith("o2") or ("diffuse" in lname) or ("continuum" in lname):
            gname = "diffuse"
            print(f"Coefficient '{name}' was assigned to 'diffuse' group.")
        elif lname.startswith("atom"):
            gname = "atomic"
            print(f"Coefficient '{name}' was assigned to 'atomic' group.")
        else:
            raise ValueError(f"Unrecognized coefficient name '{name}'")
        group_ids[i] = group_to_id[gname]
        counts[gname] += 1

    print("Coefficient group counts:", ", ".join([f"{name}={counts[name]}" for name in group_names]))
    return group_ids, group_names


class RobustScaler:
    """Median/IQR scaler with safe fallback for near-constant columns.

    This is intentionally robust to heavy-tailed coefficient distributions.
    """

    def fit(self, x):
        """Estimate per-column median and IQR scale."""
        self.med_ = np.nanmedian(x, axis=0)
        q25 = np.nanpercentile(x, 25, axis=0)
        q75 = np.nanpercentile(x, 75, axis=0)
        iqr = q75 - q25
        self.scale_ = np.where(iqr > 1e-8, iqr, 1.0)
        return self

    def transform(self, x):
        """Normalize features using fitted median and IQR."""
        return (x - self.med_) / self.scale_

    def inverse_transform(self, x):
        """Map normalized features back to original space."""
        return x * self.scale_ + self.med_


def _coef_to_model_space(coef):
    """Map physical non-negative coefficients to model space.

    The square-root transform reduces dynamic range and stabilizes optimization.
    """
    return np.sqrt(np.clip(coef, 0.0, None)).astype(np.float32)


def _coef_from_model_space(coef_model):
    """Inverse of `_coef_to_model_space` (clip + square)."""
    coef_model = np.clip(np.asarray(coef_model, dtype=np.float32), 0.0, None)
    return np.square(coef_model).astype(np.float32)




def _predict_test_coefficients_from_artifacts(artifacts_local, coef_mat_local, ctx_mat_local, test_idx_local):
    model_local = artifacts_local["model"]
    coef_scaler_local = artifacts_local["coef_scaler"]
    ctx_scaler_local = artifacts_local["ctx_scaler"]
    device_local = artifacts_local["device"]

    coef_test_model_local = _coef_to_model_space(coef_mat_local[test_idx_local])
    coef_test_n_local = coef_scaler_local.transform(coef_test_model_local).astype(np.float32)
    ctx_test_n_local = ctx_scaler_local.transform(ctx_mat_local[test_idx_local]).astype(np.float32)

    with torch.no_grad():
        coef_t_local = torch.from_numpy(coef_test_n_local).to(device_local)
        ctx_t_local = torch.from_numpy(ctx_test_n_local).to(device_local)
        mu_t_local, _ = model_local.prior(ctx_t_local)
        coef_hat_t_local = model_local.decode(mu_t_local, ctx_t_local)

    coef_hat_model_local = coef_scaler_local.inverse_transform(coef_hat_t_local.cpu().numpy())
    return _coef_from_model_space(coef_hat_model_local)


# --- In-model correlated spline + OH heads (replaces post-hoc block corrections) ---

def _extract_component_indices_from_names(coef_names_local, prefixes):
    idx = []
    pfx_l = tuple(str(p).lower() for p in prefixes)
    for i, name in enumerate(coef_names_local):
        lname = str(name).lower()
        if any(lname.startswith(pfx) for pfx in pfx_l):
            idx.append(i)
    return np.asarray(idx, dtype=np.int64)


class ConditionalVAE(nn.Module):
    """Conditional VAE with low-rank spline/OH heads and a context-only continuum path."""

    def __init__(
        self,
        n_coef,
        n_ctx,
        z_dim=8,
        hidden=256,
        coef_group_ids=None,
        n_groups=5,
        group_emb_dim=8,
        spline_indices=None,
        spline_rank=4,
        oh_indices=None,
        oh_rank=8,
        continuum_indices=None,
        continuum_rank=8,
    ):
        super().__init__()
        enc_in = n_coef + n_ctx
        dec_in = z_dim + n_ctx

        if coef_group_ids is None:
            coef_group_ids = torch.zeros(n_coef, dtype=torch.long)
        if coef_group_ids.numel() != n_coef:
            raise ValueError(f"coef_group_ids length mismatch: expected {n_coef}, got {coef_group_ids.numel()}")
        self.register_buffer("coef_group_ids", coef_group_ids.long())

        self.group_emb = nn.Embedding(n_groups, group_emb_dim)
        self.enc_group_scale = nn.Linear(group_emb_dim, 1)
        self.enc_group_bias = nn.Linear(group_emb_dim, 1)
        self.dec_group_bias = nn.Linear(group_emb_dim, 1)

        self.encoder = nn.Sequential(
            nn.Linear(enc_in, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
        )
        self.mu = nn.Linear(hidden, z_dim)
        self.logvar = nn.Linear(hidden, z_dim)

        # Learned conditional prior p(z | context) used at metadata-only inference time.
        self.prior_backbone = nn.Sequential(
            nn.Linear(n_ctx, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
        )
        self.prior_mu = nn.Linear(hidden, z_dim)
        self.prior_logvar = nn.Linear(hidden, z_dim)

        self.decoder_backbone = nn.Sequential(
            nn.Linear(dec_in, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
        )
        self.decoder_out = nn.Linear(hidden, n_coef)
        # This deterministic context head gives the metadata-only prediction path
        # enough capacity to explain coefficients without relying on the latent code.
        self.ctx_global_head = nn.Sequential(
            nn.Linear(n_ctx, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, n_coef),
        )

        if spline_indices is None:
            spline_indices = np.array([], dtype=np.int64)
        spline_indices = np.asarray(spline_indices, dtype=np.int64)
        self.register_buffer("spline_idx", torch.from_numpy(spline_indices).long())
        self.spline_rank = int(max(1, spline_rank))
        self.has_spline_head = int(spline_indices.size) >= 3
        if self.has_spline_head:
            n_spline = int(spline_indices.size)
            self.spline_factor = nn.Linear(hidden, self.spline_rank)
            self.spline_basis = nn.Parameter(0.01 * torch.randn(self.spline_rank, n_spline))

        if oh_indices is None:
            oh_indices = np.array([], dtype=np.int64)
        oh_indices = np.asarray(oh_indices, dtype=np.int64)
        self.register_buffer("oh_idx", torch.from_numpy(oh_indices).long())
        self.oh_rank = int(max(1, oh_rank))
        self.has_oh_head = int(oh_indices.size) >= 3
        if self.has_oh_head:
            n_oh = int(oh_indices.size)
            self.oh_factor = nn.Linear(hidden, self.oh_rank)
            self.oh_basis = nn.Parameter(0.01 * torch.randn(self.oh_rank, n_oh))

        if continuum_indices is None:
            continuum_indices = np.array([], dtype=np.int64)
        continuum_indices = np.asarray(continuum_indices, dtype=np.int64)
        self.register_buffer("continuum_idx", torch.from_numpy(continuum_indices).long())
        self.continuum_rank = int(max(1, continuum_rank))
        self.has_continuum_ctx_head = int(continuum_indices.size) >= 1
        if self.has_continuum_ctx_head:
            n_cont = int(continuum_indices.size)
            self.ctx_continuum_backbone = nn.Sequential(
                nn.Linear(n_ctx, hidden // 2),
                nn.GELU(),
                nn.Linear(hidden // 2, self.continuum_rank),
            )
            self.continuum_basis = nn.Parameter(0.01 * torch.randn(self.continuum_rank, n_cont))

    def _group_embedding(self):
        return self.group_emb(self.coef_group_ids)

    def _augment_coef_for_encode(self, coef):
        emb = self._group_embedding()
        scale = self.enc_group_scale(emb).squeeze(-1)
        bias = self.enc_group_bias(emb).squeeze(-1)
        return coef * (1.0 + scale.unsqueeze(0)) + bias.unsqueeze(0)

    def encode(self, coef, ctx):
        coef_aug = self._augment_coef_for_encode(coef)
        h = self.encoder(torch.cat([coef_aug, ctx], dim=-1))
        return self.mu(h), self.logvar(h)

    def prior(self, ctx):
        h = self.prior_backbone(ctx)
        return self.prior_mu(h), self.prior_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z, ctx):
        h = self.decoder_backbone(torch.cat([z, ctx], dim=-1))
        raw = self.decoder_out(h)
        raw = raw + self.ctx_global_head(ctx)
        emb = self._group_embedding()
        dec_bias = self.dec_group_bias(emb).squeeze(-1)
        raw = raw + dec_bias.unsqueeze(0)

        if self.has_spline_head:
            spline_delta = self.spline_factor(h) @ self.spline_basis
            raw[:, self.spline_idx] = raw[:, self.spline_idx] + spline_delta
        if self.has_oh_head:
            oh_delta = self.oh_factor(h) @ self.oh_basis
            raw[:, self.oh_idx] = raw[:, self.oh_idx] + oh_delta
        if self.has_continuum_ctx_head:
            continuum_delta = self.ctx_continuum_backbone(ctx) @ self.continuum_basis
            raw[:, self.continuum_idx] = raw[:, self.continuum_idx] + continuum_delta
        return raw

    def forward(self, coef, ctx):
        mu, logvar = self.encode(coef, ctx)
        z = self.reparameterize(mu, logvar)
        coef_hat = self.decode(z, ctx)
        return coef_hat, mu, logvar


def _second_difference_penalty(x):
    if x.shape[1] < 3:
        return x.new_zeros(())
    d2 = x[:, 2:] - 2.0 * x[:, 1:-1] + x[:, :-2]
    return torch.mean(d2 * d2)


def cvae_loss_stable(
    coef_hat,
    coef_true,
    mu,
    logvar,
    beta=1.0,
    latent_l2_weight=0.0,
    coef_weights=None,
    upper_bounds=None,
    upper_weight=0.0,
    spline_idx=None,
    smoothness_weight=0.0,
    oh_idx=None,
    oh_smoothness_weight=0.0,
    coef_hat_ctx=None,
    continuum_idx=None,
    context_pred_weight=0.0,
    continuum_pred_weight=0.0,
    prior_mu=None,
    prior_logvar=None,
    oh_tail_weight=0.0,
    oh_tail_gamma=2.0,
    oh_tail_ref=None,
):
    recon_raw = F.smooth_l1_loss(coef_hat, coef_true, reduction="none")
    if coef_weights is not None:
        weights = coef_weights.to(coef_true.device).view(1, -1)
        recon = (recon_raw * weights).sum() / (weights.sum() * coef_true.shape[0])
    else:
        recon = recon_raw.mean()

    latent_penalty = latent_l2_weight * mu.pow(2).mean() if latent_l2_weight > 0.0 else 0.0

    upper_penalty = 0.0
    if upper_bounds is not None and upper_weight > 0.0:
        bounds = upper_bounds.to(coef_hat.device).view(1, -1)
        upper_penalty = torch.square(F.relu(coef_hat - bounds)).mean()

    spline_smooth_penalty = 0.0
    if smoothness_weight > 0.0 and spline_idx is not None and spline_idx.numel() >= 3:
        spline_hat = coef_hat[:, spline_idx]
        spline_smooth_penalty = _second_difference_penalty(spline_hat)

    oh_smooth_penalty = 0.0
    if oh_smoothness_weight > 0.0 and oh_idx is not None and oh_idx.numel() >= 3:
        oh_hat = coef_hat[:, oh_idx]
        oh_smooth_penalty = _second_difference_penalty(oh_hat)

    oh_tail_penalty = 0.0
    if (
        oh_tail_weight > 0.0
        and oh_idx is not None
        and oh_idx.numel() > 0
        and oh_tail_ref is not None
    ):
        oh_true = coef_true[:, oh_idx]
        oh_pred = coef_hat[:, oh_idx]
        oh_ref = oh_tail_ref.to(coef_true.device).view(1, -1)
        oh_excess = F.relu(oh_true - oh_ref)
        oh_weights = 1.0 + oh_tail_gamma * oh_excess
        oh_tail_penalty = (F.smooth_l1_loss(oh_pred, oh_true, reduction="none") * oh_weights).mean()

    ctx_pred_penalty = 0.0
    continuum_ctx_penalty = 0.0
    if coef_hat_ctx is not None:
        ctx_recon_raw = F.smooth_l1_loss(coef_hat_ctx, coef_true, reduction="none")
        if context_pred_weight > 0.0:
            ctx_pred_penalty = ctx_recon_raw.mean()
        if continuum_pred_weight > 0.0 and continuum_idx is not None and continuum_idx.numel() > 0:
            continuum_ctx_penalty = ctx_recon_raw[:, continuum_idx].mean()

    logvar_c = torch.clamp(logvar, -12.0, 12.0)
    kl = -0.5 * torch.mean(1.0 + logvar_c - mu.pow(2) - logvar_c.exp())
    loss = (
        recon
        + beta * kl
        + latent_penalty
        + upper_weight * upper_penalty
        + smoothness_weight * spline_smooth_penalty
        + oh_smoothness_weight * oh_smooth_penalty
        + oh_tail_weight * oh_tail_penalty
        + context_pred_weight * ctx_pred_penalty
        + continuum_pred_weight * continuum_ctx_penalty
    )

    spline_out = spline_smooth_penalty.detach() if isinstance(spline_smooth_penalty, torch.Tensor) else torch.tensor(0.0, device=coef_hat.device)
    oh_out = oh_smooth_penalty.detach() if isinstance(oh_smooth_penalty, torch.Tensor) else torch.tensor(0.0, device=coef_hat.device)
    ctx_out = (
        (ctx_pred_penalty.detach() if isinstance(ctx_pred_penalty, torch.Tensor) else torch.tensor(0.0, device=coef_hat.device))
        + (continuum_ctx_penalty.detach() if isinstance(continuum_ctx_penalty, torch.Tensor) else torch.tensor(0.0, device=coef_hat.device))
    )
    return loss, recon.detach(), kl.detach(), spline_out, oh_out, ctx_out


def train_cvae_stable(
    coef,
    ctx,
    coef_names=None,
    z_dim=8,
    hidden=256,
    lr=1e-3,
    batch_size=256,
    n_epochs=120,
    beta_max=0.3,
    beta_warmup_epochs=40,
    latent_l2_weight=2e-4,
    grad_clip=1.0,
    seed=42,
    spline_rank=4,
    smoothness_weight=2e-4,
    spline_prefixes=("moon_bs",),
    oh_rank=8,
    oh_smoothness_weight=0.0,
    oh_prefixes=("oh_",),
    oh_tail_weight=0.20,
    oh_tail_percentile=95.0,
    oh_tail_gamma=2.0,
    continuum_rank=8,
    continuum_prefixes=("moon_bs", "ho2", "feo", "o2", "diffuse", "continuum"),
    context_pred_weight=0.5,
    continuum_pred_weight=2.0,
):
    _set_reproducibility(seed=seed, deterministic=False)

    if not np.isfinite(coef).all() or not np.isfinite(ctx).all():
        raise ValueError("Input coef/ctx contain non-finite values before training.")

    coef_model = _coef_to_model_space(coef)
    _finite_report("coef(raw)", coef)
    _finite_report("coef(model)", coef_model)
    _finite_report("ctx(raw)", ctx)

    coef_weights_t = None
    upper_bounds_t = None
    coef_group_ids_t = None
    coef_group_names = ["other"]
    spline_idx_np = np.array([], dtype=np.int64)
    oh_idx_np = np.array([], dtype=np.int64)
    continuum_idx_np = np.array([], dtype=np.int64)
    if coef_names is not None:
        coef_group_ids_np, coef_group_names = _build_physics_group_ids(coef_names)
        coef_group_ids_t = torch.from_numpy(coef_group_ids_np)
        coef_weights = np.ones(coef.shape[1], dtype=np.float32)
        upper_bounds = np.full(coef.shape[1], np.inf, dtype=np.float32)
        coef_weights_t = torch.from_numpy(coef_weights)
        upper_bounds_t = torch.from_numpy(upper_bounds)
        spline_idx_np = _extract_component_indices_from_names(coef_names, prefixes=spline_prefixes)
        oh_idx_np = _extract_component_indices_from_names(coef_names, prefixes=oh_prefixes)
        continuum_idx_np = _extract_component_indices_from_names(coef_names, prefixes=continuum_prefixes)
        print(f"Spline head targets: n={spline_idx_np.size}, rank={int(spline_rank)}")
        print(f"OH head targets: n={oh_idx_np.size}, rank={int(oh_rank)}")
        print(f"Continuum context targets: n={continuum_idx_np.size}, rank={int(continuum_rank)}")

    train_idx, val_idx, test_idx = split_indices(len(coef), seed=seed)

    coef_scaler = RobustScaler().fit(coef_model[train_idx])
    ctx_scaler = RobustScaler().fit(ctx[train_idx])
    coef_n = coef_scaler.transform(coef_model).astype(np.float32)
    ctx_n = ctx_scaler.transform(ctx).astype(np.float32)

    coef_n = np.clip(coef_n, -25.0, 25.0)
    ctx_n = np.clip(ctx_n, -25.0, 25.0)
    if not np.isfinite(coef_n).all() or not np.isfinite(ctx_n).all():
        raise ValueError("Non-finite values after scaling/clipping.")

    oh_tail_ref_t = None
    if oh_idx_np.size > 0 and oh_tail_weight > 0.0:
        oh_ref = np.percentile(
            coef_n[train_idx][:, oh_idx_np],
            float(oh_tail_percentile),
            axis=0,
        ).astype(np.float32)
        oh_tail_ref_t = torch.from_numpy(oh_ref)
        print(
            f"OH-tail robust loss active: weight={oh_tail_weight:.3f}, "
            f"percentile={oh_tail_percentile:.1f}, gamma={oh_tail_gamma:.2f}, n={oh_idx_np.size}"
        )

    _finite_report("coef(norm)", coef_n)
    _finite_report("ctx(norm)", ctx_n)

    loader_rng = torch.Generator()
    loader_rng.manual_seed(seed)
    tr_loader = make_loader(coef_n, ctx_n, train_idx, batch_size=batch_size, shuffle=True, generator=loader_rng)
    va_loader = make_loader(coef_n, ctx_n, val_idx, batch_size=batch_size, shuffle=False)
    te_loader = make_loader(coef_n, ctx_n, test_idx, batch_size=batch_size, shuffle=False)

    if torch.cuda.is_available():
        device = "cuda"
    elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        device = "mps"
    else:
        device = "cpu"
    model = ConditionalVAE(
        n_coef=coef.shape[1],
        n_ctx=ctx.shape[1],
        z_dim=z_dim,
        hidden=hidden,
        coef_group_ids=coef_group_ids_t,
        n_groups=len(coef_group_names),
        group_emb_dim=8,
        spline_indices=spline_idx_np,
        spline_rank=int(spline_rank),
        oh_indices=oh_idx_np,
        oh_rank=int(oh_rank),
        continuum_indices=continuum_idx_np,
        continuum_rank=int(continuum_rank),
    ).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)

    best = {"val_loss": np.inf, "state": None, "epoch": -1}
    history = []

    for epoch in range(1, n_epochs + 1):
        model.train()
        beta = beta_max * min(1.0, epoch / max(beta_warmup_epochs, 1))

        tr_loss, tr_recon, tr_kl, tr_spline, tr_oh, tr_ctx, n_batches, n_skipped = 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0, 0
        for coef_b, ctx_b in tr_loader:
            if not torch.isfinite(coef_b).all() or not torch.isfinite(ctx_b).all():
                n_skipped += 1
                continue

            coef_b = coef_b.to(device)
            ctx_b = ctx_b.to(device)
            opt.zero_grad(set_to_none=True)

            coef_hat, mu, logvar = model(coef_b, ctx_b)
            prior_mu_b, prior_logvar_b = model.prior(ctx_b)
            coef_hat_ctx = model.decode(prior_mu_b, ctx_b)
            loss, recon, kl, spline_smooth, oh_smooth, ctx_pred = cvae_loss_stable(
                coef_hat,
                coef_b,
                mu,
                logvar,
                beta=beta,
                latent_l2_weight=latent_l2_weight,
                coef_weights=coef_weights_t,
                upper_bounds=upper_bounds_t,
                upper_weight=0.05,
                spline_idx=model.spline_idx,
                smoothness_weight=smoothness_weight,
                oh_idx=model.oh_idx,
                oh_smoothness_weight=oh_smoothness_weight,
                coef_hat_ctx=coef_hat_ctx,
                continuum_idx=model.continuum_idx,
                context_pred_weight=context_pred_weight,
                continuum_pred_weight=continuum_pred_weight,
                prior_mu=prior_mu_b,
                prior_logvar=prior_logvar_b,
                oh_tail_weight=oh_tail_weight,
                oh_tail_gamma=oh_tail_gamma,
                oh_tail_ref=oh_tail_ref_t,
            )
            if not torch.isfinite(loss):
                n_skipped += 1
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()

            tr_loss += float(loss.item())
            tr_recon += float(recon.item())
            tr_kl += float(kl.item())
            tr_spline += float(spline_smooth.item())
            tr_oh += float(oh_smooth.item())
            tr_ctx += float(ctx_pred.item())
            n_batches += 1

        if n_batches == 0:
            raise RuntimeError(
                f"All training batches were skipped at epoch {epoch}. Check input scaling/context columns."
            )

        train_metrics = {
            "loss": tr_loss / n_batches,
            "recon": tr_recon / n_batches,
            "kl": tr_kl / n_batches,
            "spline_smooth": tr_spline / n_batches,
            "oh_smooth": tr_oh / n_batches,
            "ctx_pred": tr_ctx / n_batches,
            "n_skipped": n_skipped,
        }

        model.eval()
        val_loss, val_recon, val_kl, val_spline, val_oh, val_ctx, n_val_batches = 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0
        with torch.no_grad():
            for coef_b, ctx_b in va_loader:
                coef_b = coef_b.to(device)
                ctx_b = ctx_b.to(device)
                coef_hat, mu, logvar = model(coef_b, ctx_b)
                prior_mu_b, prior_logvar_b = model.prior(ctx_b)
                coef_hat_ctx = model.decode(prior_mu_b, ctx_b)
                loss, recon, kl, spline_smooth, oh_smooth, ctx_pred = cvae_loss_stable(
                    coef_hat,
                    coef_b,
                    mu,
                    logvar,
                    beta=beta,
                    latent_l2_weight=latent_l2_weight,
                    coef_weights=coef_weights_t,
                    upper_bounds=upper_bounds_t,
                    upper_weight=0.05,
                    spline_idx=model.spline_idx,
                    smoothness_weight=smoothness_weight,
                    oh_idx=model.oh_idx,
                    oh_smoothness_weight=oh_smoothness_weight,
                    coef_hat_ctx=coef_hat_ctx,
                    continuum_idx=model.continuum_idx,
                    context_pred_weight=context_pred_weight,
                    continuum_pred_weight=continuum_pred_weight,
                    prior_mu=prior_mu_b,
                    prior_logvar=prior_logvar_b,
                    oh_tail_weight=oh_tail_weight,
                oh_tail_gamma=oh_tail_gamma,
                oh_tail_ref=oh_tail_ref_t,
            )
                if not torch.isfinite(loss):
                    continue
                val_loss += float(loss.item())
                val_recon += float(recon.item())
                val_kl += float(kl.item())
                val_spline += float(spline_smooth.item())
                val_oh += float(oh_smooth.item())
                val_ctx += float(ctx_pred.item())
                n_val_batches += 1

        if n_val_batches == 0:
            val_metrics = {"loss": np.inf, "recon": np.inf, "kl": np.inf, "spline_smooth": np.inf, "oh_smooth": np.inf, "ctx_pred": np.inf}
        else:
            val_metrics = {
                "loss": val_loss / n_val_batches,
                "recon": val_recon / n_val_batches,
                "kl": val_kl / n_val_batches,
                "spline_smooth": val_spline / n_val_batches,
                "oh_smooth": val_oh / n_val_batches,
                "ctx_pred": val_ctx / n_val_batches,
            }

        row = {
            "epoch": epoch,
            "beta": beta,
            "train_loss": train_metrics["loss"],
            "train_recon": train_metrics["recon"],
            "train_kl": train_metrics["kl"],
            "train_spline_smooth": train_metrics["spline_smooth"],
            "train_oh_smooth": train_metrics["oh_smooth"],
            "train_ctx_pred": train_metrics["ctx_pred"],
            "train_skipped": train_metrics["n_skipped"],
            "val_loss": val_metrics["loss"],
            "val_recon": val_metrics["recon"],
            "val_kl": val_metrics["kl"],
            "val_spline_smooth": val_metrics["spline_smooth"],
            "val_oh_smooth": val_metrics["oh_smooth"],
            "val_ctx_pred": val_metrics["ctx_pred"],
        }
        history.append(row)

        if val_metrics["loss"] < best["val_loss"]:
            best["val_loss"] = val_metrics["loss"]
            best["state"] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best["epoch"] = epoch

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"epoch={epoch:03d} beta={beta:.3f} "
                f"train={train_metrics['loss']:.4f} val={val_metrics['loss']:.4f} "
                f"spline={train_metrics['spline_smooth']:.5f} oh={train_metrics['oh_smooth']:.5f} ctx={train_metrics['ctx_pred']:.5f} "
                f"skipped={train_metrics['n_skipped']}"
            )

    if best["state"] is not None:
        model.load_state_dict(best["state"])

    model.eval()
    te_loss, te_recon, te_kl, te_spline, te_oh, te_ctx, n_te_batches = 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0
    with torch.no_grad():
        for coef_b, ctx_b in te_loader:
            coef_b = coef_b.to(device)
            ctx_b = ctx_b.to(device)
            coef_hat, mu, logvar = model(coef_b, ctx_b)
            prior_mu_b, prior_logvar_b = model.prior(ctx_b)
            coef_hat_ctx = model.decode(prior_mu_b, ctx_b)
            loss, recon, kl, spline_smooth, oh_smooth, ctx_pred = cvae_loss_stable(
                coef_hat,
                coef_b,
                mu,
                logvar,
                beta=beta_max,
                latent_l2_weight=latent_l2_weight,
                coef_weights=coef_weights_t,
                upper_bounds=upper_bounds_t,
                upper_weight=0.05,
                spline_idx=model.spline_idx,
                smoothness_weight=smoothness_weight,
                oh_idx=model.oh_idx,
                oh_smoothness_weight=oh_smoothness_weight,
                coef_hat_ctx=coef_hat_ctx,
                continuum_idx=model.continuum_idx,
                context_pred_weight=context_pred_weight,
                continuum_pred_weight=continuum_pred_weight,
                prior_mu=prior_mu_b,
                prior_logvar=prior_logvar_b,
                oh_tail_weight=oh_tail_weight,
                oh_tail_gamma=oh_tail_gamma,
                oh_tail_ref=oh_tail_ref_t,
            )
            if not torch.isfinite(loss):
                continue
            te_loss += float(loss.item())
            te_recon += float(recon.item())
            te_kl += float(kl.item())
            te_spline += float(spline_smooth.item())
            te_oh += float(oh_smooth.item())
            te_ctx += float(ctx_pred.item())
            n_te_batches += 1

    if n_te_batches > 0:
        test_metrics = {
            "loss": te_loss / n_te_batches,
            "recon": te_recon / n_te_batches,
            "kl": te_kl / n_te_batches,
            "spline_smooth": te_spline / n_te_batches,
            "oh_smooth": te_oh / n_te_batches,
            "ctx_pred": te_ctx / n_te_batches,
        }
    else:
        test_metrics = {"loss": np.inf, "recon": np.inf, "kl": np.inf, "spline_smooth": np.inf, "oh_smooth": np.inf, "ctx_pred": np.inf}

    print(f"Best epoch: {best['epoch']} | best val loss: {best['val_loss']:.4f}")
    print(
        "Test metrics: "
        f"loss={test_metrics['loss']:.4f} recon={test_metrics['recon']:.4f} "
        f"kl={test_metrics['kl']:.4f} spline={test_metrics['spline_smooth']:.5f} oh={test_metrics['oh_smooth']:.5f} ctx={test_metrics['ctx_pred']:.5f}"
    )

    artifacts = {
        "model": model,
        "coef_scaler": coef_scaler,
        "ctx_scaler": ctx_scaler,
        "history": history,
        "train_idx": train_idx,
        "val_idx": val_idx,
        "test_idx": test_idx,
        "device": device,
        "coef_group_names": coef_group_names,
        "ctx_names": ctx_names,
        "spline_idx": spline_idx_np,
        "spline_rank": int(spline_rank),
        "smoothness_weight": float(smoothness_weight),
        "oh_idx": oh_idx_np,
        "oh_rank": int(oh_rank),
        "oh_smoothness_weight": float(oh_smoothness_weight),
        "continuum_idx": continuum_idx_np,
        "continuum_rank": int(continuum_rank),
    }
    return artifacts


# ---- Run stable low-beta model on joined sky1 + sky2 decomposition files ----
input_file = "lvmsframe_median_stack_1.2.1_meta_only.fits"
decomp_specs = [
    ("sky1", "lvmsframe_median_stack_1.2.1_sky1_meta_coef.fits"),
    ("sky2", "lvmsframe_median_stack_1.2.1_sky2_meta_coef.fits"),
]
context_cols = [
    "alt",
    "moon_sep",
    "moon_alt",
    "sun_alt",
    "moon_illum",
    "airmass",
]
joined_coef = []
joined_ctx = []
joined_chi2 = []
coef_names_ref = None
ctx_names_ref = None
for kind, decomp_path in decomp_specs:
    coef_k, ctx_k, coef_names_k, ctx_names_k, chi2_k = read_decomp_dataset(
        decomp_fits_path=decomp_path,
        input_fits_path=input_file,
        context_columns=context_cols,
        decomp_kind=kind,
        return_chi2=True,
    )
    if coef_names_ref is None:
        coef_names_ref = coef_names_k
        ctx_names_ref = ctx_names_k
    else:
        if coef_names_k != coef_names_ref:
            raise ValueError(f"Coefficient names mismatch for {kind}")
        if ctx_names_k != ctx_names_ref:
            raise ValueError(f"Context names mismatch for {kind}")
    joined_coef.append(coef_k)
    joined_ctx.append(ctx_k)
    joined_chi2.append(chi2_k)
coef_all = np.vstack(joined_coef)
ctx_all = np.vstack(joined_ctx)
chi2_used = np.concatenate(joined_chi2)
coef_names = coef_names_ref
ctx_names = ctx_names_ref

# ---- Optional pre-filter thinning controls ----
THIN_EVERY_N = 2  # Keep every nth row after loading; set >1 to thin.
if THIN_EVERY_N > 1:
    keep = slice(None, None, THIN_EVERY_N)
    coef_all = coef_all[keep]
    ctx_all = ctx_all[keep]
    chi2_used = chi2_used[keep]
    print(
        f"Pre-chi2 thinning: every {THIN_EVERY_N} row kept -> "
        f"n_rows={coef_all.shape[0]}"
    )
else:
    print(f"Pre-chi2 thinning disabled: n_rows={coef_all.shape[0]}")

# ---- Chi2 filter controls ----
CHI2_MIN = 0.0
CHI2_MAX = 10.0
CHI2_QMAX = 90.0
chi2_hi = np.nanpercentile(chi2_used[np.isfinite(chi2_used)], CHI2_QMAX)
chi2_upper = chi2_hi if CHI2_MAX is None else min(CHI2_MAX, chi2_hi)
mask = (
    np.isfinite(chi2_used)
    & (chi2_used >= CHI2_MIN)
    & (chi2_used <= chi2_upper)
)
print(
    f"Joined sky1+sky2 chi2 filter: min={CHI2_MIN:.3g}, qmax={CHI2_QMAX:.1f}%=>{chi2_hi:.3g}, "
    f"upper={chi2_upper:.3g} | keep={mask.sum()}/{len(mask)} "
    f"({100.0*mask.mean():.1f}%)"
)
fig_chi2_joined = px.histogram(
    x=chi2_used[mask],
    nbins=80,
    title="Joined sky1+sky2 reduced chi2 distribution (rows used for training)",
    labels={"x": "reduced chi2", "y": "count"},
)
fig_chi2_joined.update_layout(template="plotly_white", bargap=0.03)
fig_chi2_joined.show()
coef_mat = coef_all[mask]
ctx_mat = ctx_all[mask]
chi2_filtered = chi2_used[mask]

# Hard manual coefficient bounds for known problematic tails.
HARD_COEF_BOUNDS = {
    "feo": (0.0, 0.01),
    "atom_k": (0.0, 0.01),
}
coef_name_l = [str(n).lower() for n in coef_names]
manual_keep = np.ones(coef_mat.shape[0], dtype=bool)
for cname, (lo, hi) in HARD_COEF_BOUNDS.items():
    idxs = np.where(np.array(coef_name_l) == cname)[0]
    if idxs.size == 0:
        print(f"Manual hard clip: coefficient {cname} not found; skipping.")
        continue
    vals = coef_mat[:, idxs[0]]
    within = np.isfinite(vals) & (vals >= lo) & (vals <= hi)
    manual_keep &= within
    print(
        f"Manual hard clip {cname}: [{lo:.3g}, {hi:.3g}] | kept {within.sum()}/{within.size} "
        f"({100.0 * within.mean():.1f}%)"
    )
n_manual0 = coef_mat.shape[0]
coef_mat = coef_mat[manual_keep]
ctx_mat = ctx_mat[manual_keep]
chi2_filtered = chi2_filtered[manual_keep]
print(
    f"Manual hard clip combined: kept {manual_keep.sum()}/{n_manual0} "
    f"({100.0 * manual_keep.mean():.1f}%)"
)


def _kappa_sigma_row_mask(x, kappa=5.0, n_iter=3):
    """Return row mask via iterative per-feature kappa-sigma clipping."""
    x = np.asarray(x, dtype=np.float64)
    keep = np.isfinite(x).all(axis=1)
    if not np.any(keep):
        return keep
    for _ in range(n_iter):
        mu = np.nanmean(x[keep], axis=0)
        sig = np.nanstd(x[keep], axis=0)
        sig = np.where(np.isfinite(sig) & (sig > 0), sig, 1.0)
        within = np.all(np.abs(x - mu) <= (kappa * sig), axis=1)
        within &= np.isfinite(x).all(axis=1)
        new_keep = keep & within
        if new_keep.sum() == keep.sum() or new_keep.sum() == 0:
            break
        keep = new_keep
    return keep


KAPPA = 6.0
mask_coef = _kappa_sigma_row_mask(coef_mat, kappa=KAPPA, n_iter=3)
mask_kappa = mask_coef
n0 = coef_mat.shape[0]
coef_mat = coef_mat[mask_kappa]
ctx_mat = ctx_mat[mask_kappa]
chi2_filtered = chi2_filtered[mask_kappa]
print(
    f"Kappa-sigma filter (kappa={KAPPA:.1f}): kept {mask_kappa.sum()}/{n0} "
    f"({100.0 * mask_kappa.mean():.1f}%)"
)
print(f"Post-kappa shapes: coef_mat={coef_mat.shape}, ctx_mat={ctx_mat.shape}")

print(f"Loaded {coef_mat.shape[0]} filtered samples | n_coef={coef_mat.shape[1]} | n_ctx={ctx_mat.shape[1]}")
print("Context columns:", ctx_names)

artifacts = train_cvae_stable(
    coef_mat,
    ctx_mat,
    coef_names=coef_names,
    z_dim=8,
    hidden=256,
    lr=1e-3,
    batch_size=256,
    n_epochs=200,
    beta_max=0.3,
    beta_warmup_epochs=80,
    latent_l2_weight=2e-4,
    grad_clip=1.0,
    spline_rank=4,
    smoothness_weight=2e-4,
    spline_prefixes=("moon_bs",),
    oh_rank=8,
    oh_smoothness_weight=0.0,
    oh_prefixes=("oh_",),
    oh_tail_weight=0.20,
    oh_tail_percentile=95.0,
    oh_tail_gamma=2.0,
    continuum_rank=8,
    continuum_prefixes=("moon_bs", "ho2", "feo", "o2", "diffuse", "continuum"),
)
model = artifacts["model"]
model.eval()
device = artifacts["device"]
train_idx = artifacts["train_idx"]
test_idx = artifacts["test_idx"]
coef_scaler = artifacts["coef_scaler"]
ctx_scaler = artifacts["ctx_scaler"]

coef_train_model = _coef_to_model_space(coef_mat[train_idx])
coef_test_model = _coef_to_model_space(coef_mat[test_idx])
coef_train = coef_scaler.transform(coef_train_model).astype(np.float32)
coef_test = coef_scaler.transform(coef_test_model).astype(np.float32)
ctx_train = ctx_scaler.transform(ctx_mat[train_idx]).astype(np.float32)
ctx_test = ctx_scaler.transform(ctx_mat[test_idx]).astype(np.float32)

# Train-set quantile bounds to suppress latent and output outliers during inference.
z_q_low, z_q_high = 0.1, 99.9
coef_q_low, coef_q_high = 0.01, 99.99
with torch.no_grad():
    coef_tr_t = torch.from_numpy(coef_train).to(device)
    ctx_tr_t = torch.from_numpy(ctx_train).to(device)
    mu_post_tr, _ = model.encode(coef_tr_t, ctx_tr_t)
    mu_tr, _ = model.prior(ctx_tr_t)
z_lo, z_hi = _per_dim_quantile_bounds(mu_tr.cpu().numpy(), q_low=z_q_low, q_high=z_q_high)
coef_lo, coef_hi = _per_dim_quantile_bounds(coef_train, q_low=coef_q_low, q_high=coef_q_high)
print(
    f"Outlier-control bounds (Cell 7): latent q=[{z_q_low:.1f}, {z_q_high:.1f}], "
    f"coef_n q=[{coef_q_low:.1f}, {coef_q_high:.1f}]"
)

with torch.no_grad():
    coef_t = torch.from_numpy(coef_test).to(device)
    ctx_t = torch.from_numpy(ctx_test).to(device)
    mu_post_t, _ = model.encode(coef_t, ctx_t)
    mu_t, _ = model.prior(ctx_t)

    z_lo_t = torch.from_numpy(z_lo).to(device)
    z_hi_t = torch.from_numpy(z_hi).to(device)
    z_cap_width = 0.05 * torch.clamp(z_hi_t - z_lo_t, min=1e-6)
    z_excess_hi = F.softplus((mu_t - z_hi_t) / z_cap_width) * z_cap_width
    z_excess_lo = F.softplus((z_lo_t - mu_t) / z_cap_width) * z_cap_width
    mu_t_clip = mu_t - z_excess_hi + z_excess_lo

    coef_hat_t_raw = model.decode(mu_t_clip, ctx_t)

    coef_lo_t = torch.from_numpy(coef_lo).to(device)
    coef_hi_t = torch.from_numpy(coef_hi).to(device)
    cap_width = 0.05 * torch.clamp(coef_hi_t - coef_lo_t, min=1e-6)
    cap_excess = F.softplus((coef_hat_t_raw - coef_hi_t) / cap_width) * cap_width
    coef_hat_t = coef_hat_t_raw - cap_excess

z_clip_frac = float(((mu_t > z_hi_t) | (mu_t < z_lo_t)).float().mean().item())
z_soft_adjust_frac = float((torch.abs(mu_t_clip - mu_t) > 1e-7).float().mean().item())
coef_clip_frac = float((coef_hat_t_raw > coef_hi_t).float().mean().item())
coef_soft_adjust_frac = float((torch.abs(coef_hat_t - coef_hat_t_raw) > 1e-7).float().mean().item())
print(f"Cell 7 latent upper/lower exceedance fraction: {100.0 * z_clip_frac:.2f}%")
print(f"Cell 7 latent soft-cap adjustment fraction: {100.0 * z_soft_adjust_frac:.2f}%")
print(f"Cell 7 decoded-output upper-cap exceedance fraction: {100.0 * coef_clip_frac:.2f}%")
print(f"Cell 7 decoded-output soft-cap adjustment fraction: {100.0 * coef_soft_adjust_frac:.2f}%")

coef_hat_model = coef_scaler.inverse_transform(coef_hat_t.cpu().numpy())
coef_hat_phys = _coef_from_model_space(coef_hat_model)
print("Reconstruction array shape:", coef_hat_phys.shape)


def _predict_rows_from_artifacts(artifacts_local, coef_mat_local, ctx_mat_local, row_idx_local):
    model_local = artifacts_local["model"]
    coef_scaler_local = artifacts_local["coef_scaler"]
    ctx_scaler_local = artifacts_local["ctx_scaler"]
    device_local = artifacts_local["device"]

    coef_model_local = _coef_to_model_space(coef_mat_local[row_idx_local])
    coef_n_local = coef_scaler_local.transform(coef_model_local).astype(np.float32)
    ctx_n_local = ctx_scaler_local.transform(ctx_mat_local[row_idx_local]).astype(np.float32)

    with torch.no_grad():
        coef_t_local = torch.from_numpy(coef_n_local).to(device_local)
        ctx_t_local = torch.from_numpy(ctx_n_local).to(device_local)
        mu_t_local, _ = model_local.encode(coef_t_local, ctx_t_local)
        coef_hat_t_local = model_local.decode(mu_t_local, ctx_t_local)

    coef_hat_model_local = coef_scaler_local.inverse_transform(coef_hat_t_local.cpu().numpy())
    return _coef_from_model_space(coef_hat_model_local)


def apply_lowbeta_oh_correction(coef_pred_phys, ctx_phys):
    """Compatibility no-op: OH corrections are now learned in-model."""
    coef_pred_phys = np.asarray(coef_pred_phys, dtype=np.float32)
    if coef_pred_phys.ndim == 1:
        coef_pred_phys = coef_pred_phys[None, :]
    return coef_pred_phys


coef_pred_train_base = _predict_rows_from_artifacts(artifacts, coef_mat, ctx_mat, train_idx)
coef_pred_test_base = _predict_rows_from_artifacts(artifacts, coef_mat, ctx_mat, test_idx)
coef_true_train = coef_mat[train_idx].astype(np.float32)
coef_true_test = coef_mat[test_idx].astype(np.float32)
coef_hat_phys_stage1 = coef_pred_test_base.copy()
coef_hat_phys = coef_pred_test_base.copy()
coef_pred_test_corrected = coef_hat_phys.copy()
coef_test_phys = coef_true_test
print("In-model OH+spline heads active. Post-hoc OH correction disabled.")
print("Reconstruction array shape:", coef_hat_phys.shape)



Pre-chi2 thinning: every 2 row kept -> n_rows=17260
Joined sky1+sky2 chi2 filter: min=0, qmax=90.0%=>4.69, upper=4.69 | keep=15534/17260 (90.0%)


Manual hard clip feo: [0, 0.01] | kept 12828/15534 (82.6%)
Manual hard clip atom_k: [0, 0.01] | kept 12213/15534 (78.6%)
Manual hard clip combined: kept 10570/15534 (68.0%)
Kappa-sigma filter (kappa=6.0): kept 6474/10570 (61.2%)
Post-kappa shapes: coef_mat=(6474, 442), ctx_mat=(6474, 6)
Loaded 6474 filtered samples | n_coef=442 | n_ctx=6
Context columns: ['alt', 'moon_sep', 'moon_alt', 'sun_alt', 'moon_illum', 'airmass']
coef(raw): shape=(6474, 442), finite=100.00%
coef(model): shape=(6474, 442), finite=100.00%
ctx(raw): shape=(6474, 6), finite=100.00%
Coefficient 'HO2' was assigned to 'diffuse' group.
Coefficient 'FeO' was assigned to 'diffuse' group.
Coefficient 'O2Ac' was assigned to 'diffuse' group.
Coefficient 'ATOM_K' was assigned to 'atomic' group.
Coefficient 'ATOM_N' was assigned to 'atomic' group.
Coefficient 'ATOM_Na' was assigned to 'atomic' group.
Coefficient 'ATOM_Og' was assigned to 'atomic' group.
Coefficient 'ATOM_Or' was assigned to 'atomic' group.
Coefficient 'ATOM_O

In [26]:
# Diagnostics: coefficient-wise metrics + latent diagnostics
import numpy as np
import pandas as pd
from astropy.table import Table
import plotly.express as px

required = [
    "artifacts", "coef_mat", "ctx_mat", "coef_names",
    "coef_hat_phys", "mu_t_clip"
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Run Cell 7 first. Missing variables: " + ", ".join(missing)
    )

model = artifacts["model"]
model.eval()

device = artifacts["device"]
test_idx = artifacts["test_idx"]
coef_scaler = artifacts["coef_scaler"]
ctx_scaler = artifacts["ctx_scaler"]

# Use training-time context names when available; avoid global overwrite from later cells.
ctx_names_train = artifacts.get("ctx_names", None)
if ctx_names_train is None:
    if "context_cols" in globals() and len(context_cols) == ctx_mat.shape[1]:
        ctx_names_train = list(context_cols)
    elif "ctx_names" in globals() and len(ctx_names) == ctx_mat.shape[1]:
        ctx_names_train = list(ctx_names)
    else:
        ctx_names_train = [f"ctx_{i}" for i in range(ctx_mat.shape[1])]

coef_test_phys = coef_mat[test_idx].astype(np.float32)
ctx_test_phys = ctx_mat[test_idx].astype(np.float32)

# Consume Cell 7 outputs directly (no duplicate clipping/re-encoding here).
if coef_hat_phys.shape != coef_test_phys.shape:
    raise RuntimeError(
        f"Prediction shape mismatch. coef_hat_phys={coef_hat_phys.shape}, coef_test_phys={coef_test_phys.shape}. "
        "Re-run Cell 7 before diagnostics."
    )

if "z_clip_frac" in globals() and "coef_clip_frac" in globals():
    print(f"Latent clipping fraction (from Cell 7): {100.0 * float(z_clip_frac):.2f}%")
    print(f"Decoded-output clipping fraction (from Cell 7): {100.0 * float(coef_clip_frac):.2f}%")

# ----- Coefficient-wise metrics -----
rmse = np.sqrt(np.mean((coef_hat_phys - coef_test_phys) ** 2, axis=0))
mae = np.mean(np.abs(coef_hat_phys - coef_test_phys), axis=0)

corr = []
for j in range(coef_test_phys.shape[1]):
    x = coef_test_phys[:, j]
    y = coef_hat_phys[:, j]
    if np.std(x) < 1e-12 or np.std(y) < 1e-12:
        corr.append(np.nan)
    else:
        corr.append(float(np.corrcoef(x, y)[0, 1]))
corr = np.array(corr)

metrics_tbl = Table(
    {
        "coef_name": coef_names,
        "rmse": rmse,
        "mae": mae,
        "corr": corr,
    }
)
metrics_tbl.sort("rmse")

print("Top 15 coefficients by lowest RMSE:")
metrics_tbl[:15]

print("\nWorst 15 coefficients by RMSE:")
metrics_tbl[::-1][:15]

print("\nGlobal summary on test set:")
print(f"  mean RMSE = {np.nanmean(rmse):.5g}")
print(f"  median RMSE = {np.nanmedian(rmse):.5g}")
print(f"  mean corr = {np.nanmean(corr):.5g}")
print(f"  median corr = {np.nanmedian(corr):.5g}")

o2_mask = np.array([str(name).lower().startswith("o2") for name in coef_names])
if np.any(o2_mask):
    print("\nO2-specific summary:")
    print(f"  mean RMSE = {np.nanmean(rmse[o2_mask]):.5g}")
    print(f"  median RMSE = {np.nanmedian(rmse[o2_mask]):.5g}")
    print(f"  mean corr = {np.nanmean(corr[o2_mask]):.5g}")
    print(f"  median corr = {np.nanmedian(corr[o2_mask]):.5g}")

    o2_tbl = Table(
        {
            "coef_name": np.array(coef_names)[o2_mask],
            "rmse": rmse[o2_mask],
            "mae": mae[o2_mask],
            "corr": corr[o2_mask],
        }
    )
    o2_tbl.sort("rmse")
    print("\nO2 coefficient details:")
    o2_tbl

# ----- Robust scaling helpers for plotting -----
def _robust_bounds(arr, q_low=1.0, q_high=99.0, pad_frac=0.05):
    arr = np.asarray(arr, dtype=np.float64)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return (-1.0, 1.0)

    lo, hi = np.nanpercentile(arr, [q_low, q_high])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo = np.nanmin(arr)
        hi = np.nanmax(arr)

    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        c = float(arr[0])
        lo, hi = c - 0.5, c + 0.5

    span = max(hi - lo, 1e-6)
    pad = pad_frac * span
    return lo - pad, hi + pad

def _clip_to_bounds(arr, bounds):
    return np.clip(np.asarray(arr, dtype=np.float64), bounds[0], bounds[1])

def _set_bottom_facet_x_titles(fig, col_names, n_rows):
    """Show per-facet x-axis titles only on the bottom row."""
    n_cols = len(col_names)
    for r in range(n_rows):
        for c, cname in enumerate(col_names):
            axis_idx = r * n_cols + c + 1
            axis_name = "xaxis" if axis_idx == 1 else f"xaxis{axis_idx}"
            if axis_name in fig.layout:
                # Plotly numbers facet axes from bottom to top.
                fig.layout[axis_name].title.text = cname if r == 0 else ""

def _set_left_facet_y_titles(fig, row_titles, n_cols):
    """Set left y-axis titles per row to match output-group row labels."""
    # Plotly numbers y-axes from bottom to top across facet rows.
    ordered = list(row_titles)[::-1]
    for r, title in enumerate(ordered):
        axis_idx = r * n_cols + 1
        axis_name = "yaxis" if axis_idx == 1 else f"yaxis{axis_idx}"
        if axis_name in fig.layout:
            fig.layout[axis_name].title.text = title

# ----- Latent diagnostics -----
# Use clipped latent coordinates produced in Cell 7.
mu_np = mu_t_clip.cpu().numpy()

# Compact grid: columns are context variables and rows are latent dims.
max_latent_dims = min(4, mu_np.shape[1])
latent_cols = [f"z{k}" for k in range(max_latent_dims)]

# Downsample for responsive plotting on large test sets.
max_points = 5000
n_test = mu_np.shape[0]
if n_test > max_points:
    rng = np.random.default_rng(42)
    keep = np.sort(rng.choice(n_test, size=max_points, replace=False))
else:
    keep = np.arange(n_test)

mu_plot = mu_np[keep, :max_latent_dims]
ctx_plot = ctx_test_phys[keep]

rows = []
for k, zname in enumerate(latent_cols):
    for j, cname in enumerate(ctx_names_train):
        x_bounds = _robust_bounds(ctx_plot[:, j])
        y_bounds = _robust_bounds(mu_plot[:, k])
        rows.append(
            pd.DataFrame(
                {
                    "context_param": cname,
                    "context_value": _clip_to_bounds(ctx_plot[:, j], x_bounds),
                    "latent_value": _clip_to_bounds(mu_plot[:, k], y_bounds),
                    "latent_dim": zname,
                }
            )
        )

latent_ctx_df = pd.concat(rows, ignore_index=True)

fig_ctx = px.scatter(
    latent_ctx_df,
    x="context_value",
    y="latent_value",
    facet_col="context_param",
    facet_row="latent_dim",
    opacity=0.28,
    render_mode="webgl",
    title="Latent vs context grid (robust axis scaling)",
    labels={"latent_value": "latent value", "context_value": ""},
)

# Replace generic facet annotation text so each column/row clearly shows variable names.
fig_ctx.for_each_annotation(
    lambda a: a.update(
        text=a.text.replace("context_param=", "").replace("latent_dim=", "")
    )
)
fig_ctx.update_xaxes(matches=None)
fig_ctx.update_yaxes(matches=None)
fig_ctx.update_layout(
    template="plotly_white",
    height=max(650, 220 * max_latent_dims),
)
_set_bottom_facet_x_titles(fig_ctx, ctx_names_train, max_latent_dims)
fig_ctx.show()

# ----- Targeted output-vs-context grid -----
# Row 1: median of Moon_bs coefficients.
# Following rows: one continuum component each, excluding OH groups.
coef_name_l = [str(n).lower() for n in coef_names]

moon_bs_idx = np.array(
    [i for i, n in enumerate(coef_name_l) if n.startswith("moon_bs")],
    dtype=int,
)

def _continuum_group(name):
    # Explicitly skip OH-related groups/components.
    if "oh" in name:
        return None

    patterns = [
        ("moon", ["moon", "zodi"]),
        ("diffuse", ["diffuse"]),
        ("ho2", ["ho2", "hydroperoxyl"]),
        ("feo", ["feo", "iron"]),
        ("o2", ["o2", "oxygen"]),
        ("continuum", ["continuum"]),
    ]
    for gname, keys in patterns:
        if any(k in name for k in keys):
            return gname
    return None

comp_to_idx = {}
for i, n in enumerate(coef_name_l):
    if i in moon_bs_idx:
        continue
    g = _continuum_group(n)
    if g is None:
        continue
    comp_to_idx.setdefault(g, []).append(i)

row_groups = []
if moon_bs_idx.size > 0:
    row_groups.append(("moon_bs_median", moon_bs_idx))

def _find_prefixed_index(names, prefix, target_id):
    prefix = str(prefix).lower()
    target_num = int(target_id)
    for i, n in enumerate(names):
        lname = str(n).lower()
        if not lname.startswith(prefix):
            continue
        tail = "".join(ch for ch in lname[len(prefix):] if ch.isdigit())
        if tail and int(tail) == target_num:
            return i
    return None

for tid in [4, 12, 20]:
    idx = _find_prefixed_index(coef_names, "moon_bs", tid)
    if idx is None:
        print(f"moon_bs{tid} not found in coef_names; skipping in targeted plot.")
        continue
    row_groups.append((f"moon_bs{int(tid):02d}", np.array([idx], dtype=int)))

for tid in [100, 200, 300]:
    idx = _find_prefixed_index(coef_names, "oh", tid)
    if idx is None:
        print(f"OH{tid} not found in coef_names; skipping in targeted plot.")
        continue
    row_groups.append((f"oh{int(tid):02d}", np.array([idx], dtype=int)))

for gname in sorted(comp_to_idx.keys()):
    row_groups.append((gname, np.array(comp_to_idx[gname], dtype=int)))

if len(row_groups) == 0:
    raise RuntimeError(
        "Could not identify Moon_bs/continuum groups from coefficient names. "
        "Inspect coef_names naming conventions."
    )

out_rows = []
for row_name, idxs in row_groups:
    y_true = np.nanmedian(coef_test_phys[:, idxs], axis=1)
    y_pred = np.nanmedian(coef_hat_phys[:, idxs], axis=1)

    for j, cname in enumerate(ctx_names_train):
        x_raw = ctx_test_phys[:, j]
        x_bounds = _robust_bounds(x_raw)
        y_bounds = _robust_bounds(np.concatenate([y_true, y_pred]))

        out_rows.append(
            pd.DataFrame(
                {
                    "output_group": row_name,
                    "context_param": cname,
                    "context_value": _clip_to_bounds(x_raw, x_bounds),
                    "output_value": _clip_to_bounds(y_true, y_bounds),
                    "series": "true",
                }
            )
        )
        out_rows.append(
            pd.DataFrame(
                {
                    "output_group": row_name,
                    "context_param": cname,
                    "context_value": _clip_to_bounds(x_raw, x_bounds),
                    "output_value": _clip_to_bounds(y_pred, y_bounds),
                    "series": "learned",
                }
            )
        )

out_df = pd.concat(out_rows, ignore_index=True)

fig_out = px.scatter(
    out_df,
    x="context_value",
    y="output_value",
    color="series",
    facet_col="context_param",
    facet_row="output_group",
    opacity=0.30,
    render_mode="webgl",
    title="Targeted outputs vs context",
    color_discrete_map={"true": "#1f77b4", "learned": "#d62728"},
    labels={"output_value": "", "context_value": ""},
)
fig_out.for_each_annotation(
    lambda a: a.update(
        text=a.text.replace("context_param=", "").replace("output_group=", "")
    )
)
fig_out.update_xaxes(matches=None)
fig_out.update_yaxes(matches=None)
fig_out.update_layout(
    template="plotly_white",
    height=max(720, 220 * len(row_groups)),
)
_set_bottom_facet_x_titles(fig_out, ctx_names_train, len(row_groups))
row_titles = [name for name, _ in row_groups]
_set_left_facet_y_titles(fig_out, row_titles, len(ctx_names_train))
fig_out.show()

Latent clipping fraction (from Cell 7): 0.19%
Decoded-output clipping fraction (from Cell 7): 0.00%
Top 15 coefficients by lowest RMSE:

Worst 15 coefficients by RMSE:

Global summary on test set:
  mean RMSE = 518.53
  median RMSE = 0.046928
  mean corr = 0.42563
  median corr = 0.35724

O2-specific summary:
  mean RMSE = 0.21575
  median RMSE = 0.21575
  mean corr = 0.43161
  median corr = 0.43161

O2 coefficient details:


In [8]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import torch
from astropy.io import fits
from astropy.table import Table
from IPython.display import HTML, display

from sky_decomp.fit import reconstruct_component_spectra


def _meta_row_to_dict_upper(meta_row):
    names = list(meta_row.colnames) if hasattr(meta_row, "colnames") else list(meta_row.dtype.names)
    return {str(k).upper(): k for k in names}


def _safe_float(x):
    arr = np.asarray(x)
    if arr.size == 0:
        raise ValueError("Empty value cannot be converted to float")
    if arr.shape != ():
        arr = arr.ravel()[0]
    return float(arr)


def _read_ext_row(hdul, extname, row_index):
    if extname not in [h.name for h in hdul]:
        raise KeyError(f"Missing required extension: {extname}")
    arr = np.asarray(hdul[extname].data, dtype=float)
    if arr.ndim == 1:
        return arr
    if arr.ndim >= 2:
        if row_index < 0 or row_index >= arr.shape[0]:
            raise IndexError(f"row_index {row_index} out of range [0, {arr.shape[0]-1}] for {extname}")
        return np.asarray(arr[row_index], dtype=float)
    raise ValueError(f"Unsupported ndim={arr.ndim} for extension {extname}")


def _display_scrollable_table(tbl):
    """Render a table in a horizontally scrollable output block."""
    try:
        if hasattr(tbl, "to_pandas"):
            df = tbl.to_pandas()
        else:
            df = pd.DataFrame(tbl)
    except Exception:
        print(tbl)
        return

    html = df.to_html(index=False)
    display(
        HTML(
            "<div style='max-width:100%; overflow-x:auto; border:1px solid #ddd; padding:6px;'>"
            + html
            + "</div>"
        )
    )


def _context_from_meta_row(meta_row, ctx_names_local, mode):
    """Build one context row for mode in {'near', 'far', 'sci'} from META."""
    umap = _meta_row_to_dict_upper(meta_row)
    mode = str(mode).lower()
    if mode not in ("near", "far", "sci"):
        raise ValueError(f"Unsupported mode: {mode}")

    label = None
    if mode == "near" and "SKY_NEAR_LABEL" in umap:
        label = str(meta_row[umap["SKY_NEAR_LABEL"]]).strip().upper()
    elif mode == "far" and "SKY_FAR_LABEL" in umap:
        label = str(meta_row[umap["SKY_FAR_LABEL"]]).strip().upper()

    out = []
    for cname in ctx_names_local:
        key = str(cname).upper()

        # For science context, prefer SCI_<name> metadata explicitly.
        if mode == "sci":
            sci_key = f"SCI_{key}"
            if sci_key in umap:
                out.append(_safe_float(meta_row[umap[sci_key]]))
                continue

        if key in umap:
            out.append(_safe_float(meta_row[umap[key]]))
            continue

        skye_key = f"SKYE_{key}"
        skyw_key = f"SKYW_{key}"
        has_skye = skye_key in umap
        has_skyw = skyw_key in umap

        if has_skye and has_skyw and mode in ("near", "far"):
            v_e = _safe_float(meta_row[umap[skye_key]])
            v_w = _safe_float(meta_row[umap[skyw_key]])
            out.append(v_w if label == "SKYW" else v_e)
            continue

        raise KeyError(f"Missing context field for '{cname}' in META row")

    return np.asarray(out, dtype=np.float32)


def _predict_coef_from_context(ctx_row_phys):
    ctx_row_n = ctx_scaler.transform(ctx_row_phys[None, :]).astype(np.float32)
    with torch.no_grad():
        ctx_t = torch.from_numpy(ctx_row_n).to(device)
        mu_prior_t, _ = model.prior(ctx_t)
        coef_hat_n = model.decode(mu_prior_t, ctx_t).cpu().numpy()

    if "coef_lo" in globals() and "coef_hi" in globals():
        lo = np.asarray(coef_lo, dtype=np.float32)
        hi = np.asarray(coef_hi, dtype=np.float32)
        cap_w = 0.05 * np.clip(hi - lo, 1e-6, None)
        cap_excess = np.log1p(np.exp((coef_hat_n - hi[None, :]) / cap_w[None, :])) * cap_w[None, :]
        coef_hat_n = coef_hat_n - cap_excess

    coef_hat_model = coef_scaler.inverse_transform(coef_hat_n)
    coef_hat_phys = _coef_from_model_space(coef_hat_model)
    if "apply_lowbeta_oh_correction" in globals():
        coef_hat_phys = apply_lowbeta_oh_correction(coef_hat_phys, ctx_row_phys[None, :])
    return coef_hat_phys[0]


def _infer_base_dir_for_reconstruction():
    candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
    if "PALACE_DIR" in globals():
        try:
            p = Path(PALACE_DIR).resolve()
            candidates.extend([p, p.parent])
        except Exception:
            pass

    for cand in candidates:
        if (cand / "palace" / "PMD").exists() and (cand / "Spectre_HR_LATMOS_Meftah_V1_350_1000nm.txt").exists():
            return cand

    raise FileNotFoundError(
        "Could not infer reconstruction base_dir containing palace/PMD and solar reference file"
    )


def predict_and_plot_three_fields(filename, row_index):
    """Predict near/far/sci coefficients from metadata and compare to stored spectra."""
    path = Path(filename)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    with fits.open(path) as hdul:
        if "META" not in [h.name for h in hdul]:
            raise KeyError("Missing META extension")
        meta = Table(hdul["META"].data)

        i = int(row_index)
        if i < 0 or i >= len(meta):
            raise IndexError(f"row_index {i} out of range [0, {len(meta)-1}]")

        meta_row = meta[i]
        print("META row used for context generation:")
        _display_scrollable_table(meta[i:i + 1])

        wave_local = _read_ext_row(hdul, "WAVE", i)
        lsf_row = _read_ext_row(hdul, "LSF_SCI", i)
        flux_near = _read_ext_row(hdul, "FLUX_SKY_NEAR", i)
        flux_far = _read_ext_row(hdul, "FLUX_SKY_FAR", i)
        flux_sci = _read_ext_row(hdul, "FLUX_SCI", i)

    ctx_near = _context_from_meta_row(meta_row, ctx_names, mode="near")
    ctx_far = _context_from_meta_row(meta_row, ctx_names, mode="far")
    ctx_sci = _context_from_meta_row(meta_row, ctx_names, mode="sci")

    coef_near = _predict_coef_from_context(ctx_near)
    coef_far = _predict_coef_from_context(ctx_far)
    coef_sci = _predict_coef_from_context(ctx_sci)

    base_dir_guess = _infer_base_dir_for_reconstruction()

    comps_near = reconstruct_component_spectra(
        wave=wave_local,
        coef=coef_near,
        lsf_sigma=lsf_row / 2.35,
        n_spline_knots=25,
        base_dir=base_dir_guess,
    )
    comps_far = reconstruct_component_spectra(
        wave=wave_local,
        coef=coef_far,
        lsf_sigma=lsf_row / 2.35,
        n_spline_knots=25,
        base_dir=base_dir_guess,
    )
    comps_sci = reconstruct_component_spectra(
        wave=wave_local,
        coef=coef_sci,
        lsf_sigma=lsf_row / 2.35,
        n_spline_knots=25,
        base_dir=base_dir_guess,
    )

    fig = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.04,
        subplot_titles=("Near Sky", "Far Sky", "Science Field"),
    )

    panel_data = [
        (1, flux_near, comps_near["total"]),
        (2, flux_far, comps_far["total"]),
        (3, flux_sci, comps_sci["total"]),
    ]
    for row, flux_obs, flux_pred in panel_data:
        fig.add_trace(
            go.Scattergl(
                x=wave_local,
                y=flux_obs * FACTOR,
                mode="lines",
                name="observed" if row == 1 else None,
                showlegend=(row == 1),
                line=dict(color="#1f77b4", width=1.2),
            ),
            row=row,
            col=1,
        )
        fig.add_trace(
            go.Scattergl(
                x=wave_local,
                y=flux_pred,
                mode="lines",
                name="predicted reconstruction" if row == 1 else None,
                showlegend=(row == 1),
                line=dict(color="#d62728", width=1.2),
            ),
            row=row,
            col=1,
        )

    fig.update_yaxes(type="log", title_text="Flux", row=1, col=1)
    fig.update_yaxes(type="log", title_text="Flux", row=2, col=1)
    fig.update_yaxes(type="log", title_text="Flux", row=3, col=1)
    fig.update_xaxes(title_text="Wavelength [A]", row=3, col=1)
    fig.update_layout(
        template="plotly_white",
        height=980,
        title=f"Row {i}: predicted vs stored spectra (near/far/sci)",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
    )
    fig.show()

    return {
        "file": str(path),
        "row_index": i,
        "wave": wave_local,
        "lsf_sci": lsf_row,
        "near": {
            "context": ctx_near,
            "coef_pred": coef_near,
            "flux_obs": flux_near,
            "components": comps_near,
        },
        "far": {
            "context": ctx_far,
            "coef_pred": coef_far,
            "flux_obs": flux_far,
            "components": comps_far,
        },
        "sci": {
            "context": ctx_sci,
            "coef_pred": coef_sci,
            "flux_obs": flux_sci,
            "components": comps_sci,
        },
    }


# Example:
result = predict_and_plot_three_fields("lvmsframe_median_stack_1.2.1_every10.fits", row_index=567)

META row used for context generation:


path,exposure,expnum,mjd,tile_id,obstime,date_obs,sci_ra,sci_dec,skye_ra,skye_dec,skyw_ra,skyw_dec,sky_near_label,sky_far_label,sky_near_ra,sky_near_dec,sky_far_ra,sky_far_dec,skye_sep_deg,skyw_sep_deg,sky_near_sep_deg,sky_far_sep_deg,sci_alt,skye_alt,skyw_alt,sci_airmass,skye_airmass,skyw_airmass,sci_moon_sep,skye_moon_sep,skyw_moon_sep,moon_alt,sun_alt,moon_ra,moon_dec,moon_phase,moon_fli,moon_illum,fibers_sci,fiberfrac_sci,fibers_sky_near,fiberfrac_sky_near,fibers_sky_far,fiberfrac_sky_far,worker_pid,status,error
/data/sas/sdsswork/lvm/spectro/redux/1.2.1/1029XX/1029191/60392/lvmSFrame-00014764.fits,14764,14764,60392,1029191,2024-03-23T01:26:38.031,2024-03-23T01:26:38.031,89.264665,-0.264863,70.859412,-56.127942,88.653906,1.805877,SkyW,SkyE,88.653906,1.805877,70.859412,-56.127942,57.8144,2.1589,2.1589,57.8144,40.3703,39.7099,38.5792,1.543866,1.56519,1.603601,72.8394,100.3133,72.9973,39.9741,-34.3298,161.65505,11.91761,155.7,0.9559,0.9559,1228,0.700114,42,0.7,40,0.701754,189986,OK,


In [27]:
# Bulk diagnostic: per-row whitened residual score s_i for coefficient vectors
import numpy as np
import pandas as pd


def compute_si_statistics(coef_true, coef_pred, coef_names=None, scale="mad", eps=1e-8):
    """Compute per-row whitened residual scores s_i and summary statistics.

    Parameters
    ----------
    coef_true : array-like, shape (n_samples, n_coef)
        Reference decomposition coefficients.
    coef_pred : array-like, shape (n_samples, n_coef)
        Predicted coefficient vectors.
    coef_names : list[str] or None
        Optional coefficient names (for per-coefficient table).
    scale : {"mad", "std"}
        Per-coefficient scale used to whiten residuals.
    eps : float
        Floor added to scales for numerical stability.

    Returns
    -------
    dict with keys:
        "s_i": np.ndarray shape (n_samples,)
        "summary": dict of aggregate diagnostics
        "coef_table": pandas.DataFrame per-coefficient diagnostics
        "scales": np.ndarray shape (n_coef,)
    """
    y_true = np.asarray(coef_true, dtype=np.float64)
    y_pred = np.asarray(coef_pred, dtype=np.float64)

    if y_true.ndim != 2 or y_pred.ndim != 2:
        raise ValueError("coef_true and coef_pred must both be 2D arrays")
    if y_true.shape != y_pred.shape:
        raise ValueError(f"Shape mismatch: true={y_true.shape}, pred={y_pred.shape}")

    residual = y_pred - y_true

    if scale.lower() == "mad":
        med = np.nanmedian(residual, axis=0)
        mad = np.nanmedian(np.abs(residual - med[None, :]), axis=0)
        scales = 1.4826 * mad
    elif scale.lower() == "std":
        scales = np.nanstd(residual, axis=0)
    else:
        raise ValueError("scale must be one of {'mad', 'std'}")

    scales = np.where(np.isfinite(scales) & (scales > eps), scales, eps)

    z = residual / scales[None, :]
    s_i = np.sqrt(np.nanmean(z * z, axis=1))

    coef_rmse = np.sqrt(np.nanmean(residual * residual, axis=0))
    coef_mae = np.nanmean(np.abs(residual), axis=0)
    coef_bias = np.nanmean(residual, axis=0)

    n_coef = y_true.shape[1]
    if coef_names is None or len(coef_names) != n_coef:
        coef_names = [f"coef_{j:04d}" for j in range(n_coef)]

    coef_table = pd.DataFrame(
        {
            "coef_name": coef_names,
            "scale_used": scales,
            "bias": coef_bias,
            "mae": coef_mae,
            "rmse": coef_rmse,
        }
    ).sort_values("rmse", ascending=False, ignore_index=True)

    summary = {
        "n_samples": int(y_true.shape[0]),
        "n_coef": int(n_coef),
        "scale": scale,
        "s_i_mean": float(np.nanmean(s_i)),
        "s_i_median": float(np.nanmedian(s_i)),
        "s_i_std": float(np.nanstd(s_i)),
        "s_i_p90": float(np.nanpercentile(s_i, 90.0)),
        "s_i_p95": float(np.nanpercentile(s_i, 95.0)),
        "s_i_p99": float(np.nanpercentile(s_i, 99.0)),
        "frac_s_i_gt_2": float(np.nanmean(s_i > 2.0)),
        "frac_s_i_gt_3": float(np.nanmean(s_i > 3.0)),
    }

    return {
        "s_i": s_i,
        "summary": summary,
        "coef_table": coef_table,
        "scales": scales,
    }


# Ready-to-run example using arrays created earlier in this notebook
si_out = compute_si_statistics(
    coef_true=coef_test_phys,
    coef_pred=coef_hat_phys,
    coef_names=coef_names,
    scale="mad",
)

print("s_i summary:")
for k, v in si_out["summary"].items():
    if isinstance(v, float):
        print(f"  {k}: {v:.6g}")
    else:
        print(f"  {k}: {v}")

# Keep these handy in globals for downstream plots/filtering
s_i = si_out["s_i"]
coef_diag_table = si_out["coef_table"]

print("\nTop 10 coefficients by RMSE:")
display(coef_diag_table.head(10))

s_i summary:
  n_samples: 648
  n_coef: 442
  scale: mad
  s_i_mean: 27307.2
  s_i_median: 90.3045
  s_i_std: 81246.2
  s_i_p90: 66055.2
  s_i_p95: 152972
  s_i_p99: 449603
  frac_s_i_gt_2: 0.833333
  frac_s_i_gt_3: 0.79784

Top 10 coefficients by RMSE:


,coef_name,scale_used,bias,mae,rmse
0,OH_122,18913.273378,54664.411143,65339.867672,166107.183167
1,OH_037,4830.904419,13261.056333,15916.765230,41953.107109
2,OH_046,635.079456,2473.521884,2822.847398,8716.883623
3,OH_045,18.952044,431.535415,507.040091,2285.867973
4,OH_040,322.193485,721.745741,907.796728,2258.544329
5,OH_042,207.447719,643.611353,751.364443,1855.727688
6,OH_044,153.867959,377.531664,485.844854,1469.647449
7,OH_038,475.932698,369.190528,723.951823,1426.356420
8,OH_177,12.319407,152.278984,170.191713,443.751403
9,OH_333,34.066477,104.922522,127.142301,382.149726


In [28]:
# Latent-capacity check: how many latent dimensions are actually active?
import numpy as np
import pandas as pd

required = ["artifacts", "mu_tr", "mu_t_clip", "coef_names", "coef_hat_phys", "coef_mat"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError("Run Cell 8 first. Missing variables: " + ", ".join(missing))

test_idx_local = np.asarray(artifacts["test_idx"], dtype=int)
coef_test_phys_local = np.asarray(coef_mat[test_idx_local], dtype=np.float32)
mu_train_np = np.asarray(mu_tr.detach().cpu().numpy(), dtype=np.float64)
mu_test_np = np.asarray(mu_t_clip.detach().cpu().numpy(), dtype=np.float64)

latent_std = np.std(mu_train_np, axis=0)
latent_var = latent_std ** 2
latent_var_frac = latent_var / max(np.sum(latent_var), 1e-12)
latent_pr = float((np.sum(latent_var) ** 2) / max(np.sum(latent_var ** 2), 1e-12))

latent_tbl = pd.DataFrame(
    {
        "latent_dim": [f"z{i}" for i in range(mu_train_np.shape[1])],
        "train_std": latent_std,
        "train_var_frac": latent_var_frac,
        "test_mean_abs": np.mean(np.abs(mu_test_np), axis=0),
    }
).sort_values("train_std", ascending=False, ignore_index=True)

print(f"Configured z_dim = {mu_train_np.shape[1]}")
print(f"Latent participation ratio ~= {latent_pr:.2f}")
display(latent_tbl)

moon_mask = np.array([str(name).lower().startswith("moon_bs") for name in coef_names])
if np.any(moon_mask):
    moon_rmse = np.sqrt(np.mean((coef_hat_phys[:, moon_mask] - coef_test_phys_local[:, moon_mask]) ** 2, axis=0))
    print(f"Moon/spline median RMSE = {np.median(moon_rmse):.5g} | p90 = {np.percentile(moon_rmse, 90):.5g}")
else:
    print("No moon_bs block found in coef_names.")

Configured z_dim = 8
Latent participation ratio ~= 4.49


,latent_dim,train_std,train_var_frac,test_mean_abs
0,z6,2.118865,0.389165,1.684105
1,z2,1.388064,0.167012,1.349305
2,z5,1.233397,0.131866,1.159917
3,z1,1.104854,0.105813,0.999770
4,z7,0.997442,0.086239,0.825211
5,z0,0.977483,0.082822,1.141155
6,z4,0.478200,0.019822,0.399745
7,z3,0.446239,0.017261,0.375849


Moon/spline median RMSE = 2.0252 | p90 = 8.783


In [4]:
# Focused check: moon_alt / moon_illum correlations with continuum-like components
import numpy as np
import pandas as pd

required = ["artifacts", "coef_mat", "ctx_mat", "coef_names", "coef_hat_phys", "ctx_names"]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError("Run the main ND model cell first. Missing: " + ", ".join(missing))

ctx_names_l = [str(c).strip().lower() for c in ctx_names]
if "moon_alt" not in ctx_names_l or "moon_illum" not in ctx_names_l:
    raise RuntimeError("Context must include moon_alt and moon_illum.")

j_alt = ctx_names_l.index("moon_alt")
j_illum = ctx_names_l.index("moon_illum")

test_idx_local = np.asarray(artifacts["test_idx"], dtype=int)
ctx_test = np.asarray(ctx_mat[test_idx_local], dtype=np.float64)
coef_true = np.asarray(coef_mat[test_idx_local], dtype=np.float64)
coef_pred = np.asarray(coef_hat_phys, dtype=np.float64)

if coef_pred.shape != coef_true.shape:
    raise RuntimeError(f"Shape mismatch: pred={coef_pred.shape}, true={coef_true.shape}")

x_alt = ctx_test[:, j_alt]
x_illum = ctx_test[:, j_illum]

coef_names_l = [str(n).lower() for n in coef_names]

def _idx_starts(prefix):
    p = str(prefix).lower()
    return np.array([i for i, n in enumerate(coef_names_l) if n.startswith(p)], dtype=int)

def _idx_continuum_like():
    keep = []
    for i, n in enumerate(coef_names_l):
        is_cont = n.startswith("moon_bs") or n.startswith("ho2") or n.startswith("feo") or n.startswith("o2") or ("diffuse" in n) or ("continuum" in n)
        is_excluded = n.startswith("oh") or n.startswith("atom")
        if is_cont and not is_excluded:
            keep.append(i)
    return np.array(keep, dtype=int)

def _safe_corr(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    good = np.isfinite(a) & np.isfinite(b)
    if good.sum() < 3:
        return np.nan
    aa = a[good]
    bb = b[good]
    if np.std(aa) < 1e-12 or np.std(bb) < 1e-12:
        return np.nan
    return float(np.corrcoef(aa, bb)[0, 1])

def _group_signal(arr, idx):
    if idx.size == 0:
        return None
    return np.nanmedian(arr[:, idx], axis=1)

groups = {
    "moon_bs_median": _idx_starts("moon_bs"),
    "ho2_median": _idx_starts("ho2"),
    "feo_median": _idx_starts("feo"),
    "o2_median": _idx_starts("o2"),
    "continuum_like_median": _idx_continuum_like(),
}

rows = []
for gname, gidx in groups.items():
    if gidx.size == 0:
        continue
    y_true = _group_signal(coef_true, gidx)
    y_pred = _group_signal(coef_pred, gidx)
    rows.append({
        "group": gname,
        "n_coef": int(gidx.size),
        "corr_true_moon_alt": _safe_corr(x_alt, y_true),
        "corr_pred_moon_alt": _safe_corr(x_alt, y_pred),
        "corr_true_moon_illum": _safe_corr(x_illum, y_true),
        "corr_pred_moon_illum": _safe_corr(x_illum, y_pred),
    })

corr_df = pd.DataFrame(rows)
if corr_df.empty:
    raise RuntimeError("No continuum-like groups found in coef_names.")

for c in ["corr_true_moon_alt", "corr_pred_moon_alt", "corr_true_moon_illum", "corr_pred_moon_illum"]:
    corr_df[c + "_abs"] = np.abs(corr_df[c])

print("Moon context correlation check for continuum-like groups (test split):")
display(
    corr_df[[
        "group", "n_coef",
        "corr_true_moon_alt", "corr_pred_moon_alt",
        "corr_true_moon_illum", "corr_pred_moon_illum",
    ]].sort_values("corr_true_moon_illum", key=np.abs, ascending=False)
)

# Optional latent-context check for interpretability.
if "mu_t_clip" in globals():
    mu_np = np.asarray(mu_t_clip.detach().cpu().numpy(), dtype=np.float64)
    lat_rows = []
    for k in range(mu_np.shape[1]):
        z = mu_np[:, k]
        lat_rows.append({
            "latent_dim": f"z{k}",
            "corr_moon_alt": _safe_corr(x_alt, z),
            "corr_moon_illum": _safe_corr(x_illum, z),
        })
    lat_df = pd.DataFrame(lat_rows)
    lat_df["abs_max"] = np.maximum(np.abs(lat_df["corr_moon_alt"]), np.abs(lat_df["corr_moon_illum"]))
    print("Top latent dimensions by |corr| with moon_alt/moon_illum:")
    display(lat_df.sort_values("abs_max", ascending=False).head(8))
else:
    print("mu_t_clip not found; skipped latent-context correlation check.")

Moon context correlation check for continuum-like groups (test split):


,group,n_coef,corr_true_moon_alt,corr_pred_moon_alt,corr_true_moon_illum,corr_pred_moon_illum
4,continuum_like_median,33,0.702581,0.294661,0.593757,0.322276
0,moon_bs_median,29,0.697858,0.291176,0.592718,0.318064
3,o2_median,2,0.458983,0.322724,0.363066,0.253853
2,feo_median,1,0.232198,0.139013,0.203666,0.151625
1,ho2_median,1,0.121455,0.129280,0.105944,0.150859


Top latent dimensions by |corr| with moon_alt/moon_illum:


,latent_dim,corr_moon_alt,corr_moon_illum,abs_max
6,z6,0.910405,0.456405,0.910405
5,z5,0.890742,0.579021,0.890742
7,z7,0.887162,0.452220,0.887162
2,z2,0.875456,0.471254,0.875456
1,z1,0.820619,0.188164,0.820619
0,z0,-0.812688,-0.536628,0.812688
4,z4,0.744374,0.396493,0.744374
3,z3,-0.658039,-0.721540,0.721540


In [5]:
# ND check: effect of output soft-cap on moon_alt/moon_illum continuum correlations
import numpy as np
import pandas as pd

required = [
    "coef_hat_t_raw", "coef_hat_t", "coef_scaler", "coef_mat", "artifacts",
    "ctx_mat", "ctx_names", "coef_names", "_coef_from_model_space"
]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError("Run the main ND model cell first. Missing: " + ", ".join(missing))

ctx_names_l = [str(c).strip().lower() for c in ctx_names]
j_alt = ctx_names_l.index("moon_alt")
j_illum = ctx_names_l.index("moon_illum")

test_idx_local = np.asarray(artifacts["test_idx"], dtype=int)
ctx_test = np.asarray(ctx_mat[test_idx_local], dtype=np.float64)
coef_true = np.asarray(coef_mat[test_idx_local], dtype=np.float64)

coef_raw_model = coef_scaler.inverse_transform(coef_hat_t_raw.detach().cpu().numpy())
coef_cap_model = coef_scaler.inverse_transform(coef_hat_t.detach().cpu().numpy())
coef_raw_phys = _coef_from_model_space(coef_raw_model).astype(np.float64)
coef_cap_phys = _coef_from_model_space(coef_cap_model).astype(np.float64)

coef_names_l = [str(n).lower() for n in coef_names]
cont_idx = np.asarray([
    i for i, n in enumerate(coef_names_l)
    if (n.startswith("moon_bs") or n.startswith("ho2") or n.startswith("feo") or n.startswith("o2") or ("diffuse" in n) or ("continuum" in n))
    and (not n.startswith("oh")) and (not n.startswith("atom"))
], dtype=int)


def _safe_corr(a, b):
    good = np.isfinite(a) & np.isfinite(b)
    if good.sum() < 3:
        return np.nan
    aa = np.asarray(a[good], dtype=np.float64)
    bb = np.asarray(b[good], dtype=np.float64)
    if np.std(aa) < 1e-12 or np.std(bb) < 1e-12:
        return np.nan
    return float(np.corrcoef(aa, bb)[0, 1])

x_alt = ctx_test[:, j_alt]
x_illum = ctx_test[:, j_illum]
y_true = np.nanmedian(coef_true[:, cont_idx], axis=1)
y_raw = np.nanmedian(coef_raw_phys[:, cont_idx], axis=1)
y_cap = np.nanmedian(coef_cap_phys[:, cont_idx], axis=1)

out = pd.DataFrame([
    {
        "series": "true",
        "corr_moon_alt": _safe_corr(x_alt, y_true),
        "corr_moon_illum": _safe_corr(x_illum, y_true),
    },
    {
        "series": "pred_raw_before_cap",
        "corr_moon_alt": _safe_corr(x_alt, y_raw),
        "corr_moon_illum": _safe_corr(x_illum, y_raw),
    },
    {
        "series": "pred_after_cap",
        "corr_moon_alt": _safe_corr(x_alt, y_cap),
        "corr_moon_illum": _safe_corr(x_illum, y_cap),
    },
])

print("ND continuum_like median: moon-context correlation before vs after output cap")
display(out)
if "coef_soft_adjust_frac" in globals():
    print(f"Decoded-output soft-cap adjustment fraction: {100.0 * float(coef_soft_adjust_frac):.2f}%")

ND continuum_like median: moon-context correlation before vs after output cap


,series,corr_moon_alt,corr_moon_illum
0,true,0.702581,0.593757
1,pred_raw_before_cap,0.706667,0.596357
2,pred_after_cap,0.706741,0.596386


Decoded-output soft-cap adjustment fraction: 78.48%


In [6]:
# Reconcile correlation arrays used in ND diagnostics
import numpy as np

ctx_names_l = [str(c).strip().lower() for c in ctx_names]
j_alt = ctx_names_l.index("moon_alt")
j_illum = ctx_names_l.index("moon_illum")

test_idx_local = np.asarray(artifacts["test_idx"], dtype=int)
ctx_test = np.asarray(ctx_mat[test_idx_local], dtype=np.float64)
coef_true = np.asarray(coef_mat[test_idx_local], dtype=np.float64)
coef_curr = np.asarray(coef_hat_phys, dtype=np.float64)

coef_names_l = [str(n).lower() for n in coef_names]
cont_idx = np.asarray([
    i for i, n in enumerate(coef_names_l)
    if (n.startswith("moon_bs") or n.startswith("ho2") or n.startswith("feo") or n.startswith("o2") or ("diffuse" in n) or ("continuum" in n))
    and (not n.startswith("oh")) and (not n.startswith("atom"))
], dtype=int)

def _safe_corr(a, b):
    g = np.isfinite(a) & np.isfinite(b)
    aa = np.asarray(a[g], dtype=np.float64)
    bb = np.asarray(b[g], dtype=np.float64)
    if aa.size < 3 or np.std(aa) < 1e-12 or np.std(bb) < 1e-12:
        return np.nan
    return float(np.corrcoef(aa, bb)[0, 1])

x_alt = ctx_test[:, j_alt]
x_illum = ctx_test[:, j_illum]
y_true = np.nanmedian(coef_true[:, cont_idx], axis=1)
y_curr = np.nanmedian(coef_curr[:, cont_idx], axis=1)

print("coef_hat_phys shape:", coef_curr.shape)
print("corr true moon_alt:", _safe_corr(x_alt, y_true))
print("corr pred(coef_hat_phys) moon_alt:", _safe_corr(x_alt, y_curr))
print("corr true moon_illum:", _safe_corr(x_illum, y_true))
print("corr pred(coef_hat_phys) moon_illum:", _safe_corr(x_illum, y_curr))

coef_hat_phys shape: (648, 442)
corr true moon_alt: 0.7025814710124646
corr pred(coef_hat_phys) moon_alt: 0.2946611224132678
corr true moon_illum: 0.5937572602554471
corr pred(coef_hat_phys) moon_illum: 0.3222762904195211
